# PACT — Polynomial ANOVA Certified Transfer
### Fidelity Is a Certificate: Distilling Black-Box Teachers into Analytically Decomposable PCE–HDMR Students

**Single-file experimental notebook.** Everything needed to reproduce the
report is in this one file — no repository to clone, no package to install
beyond two pip lines. Run top to bottom.

| | |
|---|---|
| Datasets | 4 real OpenML tabular sets: `bank-marketing` (d=16), `default-of-credit-card-clients` (d=23), `gas-drift` (d=128, 6 classes), `musk v2` (d=166) |
| Seeds | 10 |
| Teacher | MLP 256–128–64, **50 epochs**, frozen (plus a GBM ablation) |
| Students | **GAS-PCE** (ours) · soft decision tree (Frosst & Hinton) · GA²M (Lou et al. / Tan et al.) |
| Reference | Monte-Carlo Sobol' on the teacher · KernelSHAP · permutation importance |

---

### What this notebook is testing

A PCE student arranged as an HDMR gives Sobol' indices in closed form from its
coefficients. The question is whether those indices describe the **teacher**.
They do, with a bound:

$$\sum_{\mathbf{u}}\bigl|S_\mathbf{u}^{\mathcal S}-S_\mathbf{u}^{\mathcal T}\bigr|
\;\le\;\frac{4\eta}{1-\eta},
\qquad \eta=\frac{\lVert f^{\mathcal T}-f^{\mathcal S}\rVert_{L^2(\mu)}}{\sigma(f^{\mathcal T})}$$

uniform over all $2^d-1$ interaction sets at once, under the **product
reference measure** $\mu$ that the ANOVA decomposition is orthogonal under.
Section 2 verifies the bound numerically before any real data is touched —
a violation there is a bug, not a finding.

> **Runtime.** Leave `QUICK = True` for a first pass (a few minutes). The full
> grid is roughly 2–4 h on a T4. Results are checkpointed per cell, so a
> disconnect costs you only the cell in flight.

## 0. Setup

In [ ]:
#@title Setup { display-mode: "form" }
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/pact_runs"  #@param {type:"string"}

import os, sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "interpret-core==0.6.*", "shap>=0.44"], check=False)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        pathlib.Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)

OUT_DIR = DRIVE_DIR if (IN_COLAB and USE_DRIVE) else "runs"
pathlib.Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

import numpy as np
print("python :", sys.version.split()[0])
try:
    import torch
    print("torch  :", torch.__version__, "| cuda:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu    :", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch  : NOT AVAILABLE —", e)
    print("         the MLP teacher and the SDT baseline will be skipped;")
    print("         set teacher='gbm' to run without torch.")
print("output :", OUT_DIR)

## 1. Implementation

The next cells are the complete implementation, inlined from the `pact/` package so this notebook stands alone. They define classes and functions only — nothing runs until §2.

### 1.1 Configuration — datasets, seeds, epochs, every hyperparameter

In [ ]:
"""Central configuration for PACT experiments.

Everything that a reviewer would want to check -- dataset identifiers, seed
count, epoch count, hyperparameter grids -- lives here and nowhere else.
"""
from __future__ import annotations

from dataclasses import dataclass, field, asdict
from typing import Sequence

# ---------------------------------------------------------------------------
# Datasets.  Four real tabular classification sets from OpenML.  None is on the
# assignment's excluded list (Iris, Titanic, Wine, Breast Cancer, Adult,
# German Credit).  Two moderate-dimensional (8-30 features), two with >= 100.
# ---------------------------------------------------------------------------


@dataclass(frozen=True)
class DatasetSpec:
    key: str
    openml_id: int
    openml_name: str          # used as a fallback if the id lookup fails
    regime: str               # "moderate" | "high-dim"
    note: str                 # why this dataset was chosen (goes in the report)


DATASETS: tuple[DatasetSpec, ...] = (
    DatasetSpec(
        key="bank",
        openml_id=1461,
        openml_name="bank-marketing",
        regime="moderate",
        note="Mixed categorical/numeric socio-economic features, mildly "
             "dependent; the benign regime for the product reference measure.",
    ),
    DatasetSpec(
        key="credit",
        openml_id=42477,
        openml_name="default-of-credit-card-clients",
        regime="moderate",
        note="Repeated-measure billing columns are near-collinear; the "
             "adversarial case for Proposition 3 at moderate d.",
    ),
    DatasetSpec(
        key="gas",
        openml_id=1476,
        openml_name="gas-drift",
        regime="high-dim",
        note="128 chemical-sensor features in correlated blocks, 6 classes; "
             "exercises the multi-output Sobol' formulation.",
    ),
    DatasetSpec(
        key="musk",
        openml_id=1116,
        openml_name="musk",
        regime="high-dim",
        note="166 features with only 6.6k samples; the regime where screening "
             "and the group lasso matter most.",
    ),
)

DATASETS_BY_KEY = {d.key: d for d in DATASETS}


# ---------------------------------------------------------------------------
# Protocol
# ---------------------------------------------------------------------------


@dataclass
class RunConfig:
    # --- protocol -----------------------------------------------------------
    seeds: Sequence[int] = tuple(range(10))     # 10 seeds
    test_size: float = 0.25
    val_size: float = 0.10

    # --- teacher ------------------------------------------------------------
    teacher: str = "mlp"                        # "mlp" | "gbm"
    epochs: int = 50                            # 50 epochs
    hidden: Sequence[int] = (256, 128, 64)
    dropout: float = 0.1
    lr: float = 1e-3
    weight_decay: float = 1e-4
    batch_size: int = 256

    # --- PCE student (GAS-PCE) ---------------------------------------------
    degree: int = 4                             # p
    q_norm: float = 0.9                         # hyperbolic truncation q
    max_order: int = 2
    screen_m: int = 32                          # top-m features get pairs
    gamma: float = 2.0                          # interaction penalty multiplier
    n_lambda: int = 20
    lambda_min_ratio: float = 1e-4
    max_iter: int = 2000
    tol: float = 1e-7

    # --- query set ----------------------------------------------------------
    rho: float = 0.5                            # product-measure query fraction
    n_query: int = 20000                        # cap on |Z|
    n_holdout_mu: int = 8000                    # held-out sample from mu for eta

    # --- reference explanations --------------------------------------------
    sobol_mc_base: int = 4096                   # Jansen base sample size
    shap_n_explain: int = 200
    shap_n_background: int = 50

    # --- baselines ----------------------------------------------------------
    # The SDT defaults are deliberate: with small gate weights and beta=1 the
    # tree cold-starts at the constant-mean predictor (eta = 1.0) and never
    # escapes in 50 epochs. See tools/diagnose_sdt.py.
    sdt_depth: int = 5
    sdt_epochs: int = 300
    sdt_beta: float = 4.0
    sdt_w_scale: float = 1.0
    sdt_lr: float = 2e-2
    ga2m_interactions: int = 20

    # --- rigour knobs -------------------------------------------------------
    # Operating point: the SMALLEST model on the lambda path within knee_tol of
    # the best held-out eta_mu. Fixed in advance and computed from eta alone,
    # so the choice cannot be tuned to flatter the certificate.
    knee_tol: float = 0.05
    # absolute slack on the knee rule, so the relative tolerance does not
    # collapse to "largest model on the path" when the best residual is tiny
    knee_abs_tol: float = 0.005
    # Within-seed bootstrap of eta-hat on the held-out mu sample -> the
    # interval on Delta(eta) that the report's Limitations section claims.
    n_bootstrap: int = 400
    # A seed whose teacher barely beats the majority class makes every
    # downstream number meaningless; flag those rather than averaging them in.
    teacher_min_lift: float = 0.02
    # The predeclared fitting criterion, applied identically to every student:
    # a student must beat the constant-mean predictor on the measure it was
    # FITTED on by this margin, or its row is invalid. eta = 1.0 is exactly the
    # constant-mean predictor, so this is "explain at least 5% of the teacher's
    # variance where you were trained". Declared here, not inside a function
    # default, so the threshold is visible to a reader and recorded in the
    # config digest. It is deliberately NOT applied to eta_mu: at rho<1 a
    # student is not fitted on mu, and eta_mu >= 1 there is Proposition 3
    # rather than a training failure.
    student_min_fit: float = 0.95
    # Fraction of the training rows reserved EXCLUSIVELY for building Z_fit,
    # the held-out sample the operating point is chosen on.
    #
    # Without this the rule is not held out at all. Z draws n_query rows from
    # X_train and Z_fit draws n_holdout_mu rows from the same pool, so on
    # gas-drift (X_train ~ 9.4k rows, n_query = 12000, hence WITH replacement)
    # and musk (~4.5k rows) the query set covers essentially every training
    # row and lambda was selected on data the student had already seen. The
    # bias is toward under-regularisation: an in-sample criterion keeps
    # rewarding the next component, so the reported model is larger and its
    # eta_fit smaller than an honest held-out criterion would give. Those two
    # datasets carry the headline rho-ablation result, so the leak sat exactly
    # where it mattered most.
    #
    # Set to 0.0 to restore the earlier (leaky) behaviour; the value enters
    # the config digest either way, so runs made under the two settings are
    # distinguishable rather than silently pooled.
    fit_holdout_frac: float = 0.2
    # Target condition number for the design. Basis columns the data does not
    # identify are dropped until the retained Gram satisfies this, because the
    # design is genuinely rank deficient on gas-drift at rho=0 (rank 1340 of
    # 1341, and the smallest RETAINED eigenvalue only 1.3x above the
    # machine-epsilon rank cutoff, so several directions are effectively null
    # and eps keeps one by luck). Coefficients along such a direction are not
    # determined by the data, so the Sobol' index read off them is arbitrary
    # and can move between BLAS builds: eps*cond ~ 6e-4 relative at 2.6e12.
    #
    # Dropping rather than ridging: a ridge makes the minimiser unique but
    # gives the stabilised direction curvature 2*eps, so FISTA needs
    # O(sqrt(1/eps)) ~ 1e4 iterations to reach it against the 800 a run uses.
    # Measured, it cut start-dependence only 12x. Dropping removes the
    # direction, keeps the retained subspace unbiased, and gives the dropped
    # terms a Sobol' index of exactly zero. See tools/test_ridge_uniqueness.py.
    # Set to inf to disable pruning.
    max_cond: float = 1e10
    # Budget-matched baselines: sweep SDT depth / GA2M interactions and report
    # the setting closest in component count to the GAS-PCE operating point.
    # Fitting a boosted GA2M is the single most expensive stage at d>100, and
    # budget matching multiplies it by the grid length, so it is restricted to
    # the moderate-dimensional sets where the fidelity comparison is actually
    # contested. Above budget_match_max_d the single default setting is used
    # and the records say so, rather than quietly claiming a matched budget.
    budget_match: bool = False
    budget_match_max_d: int = 32
    sdt_depth_grid: Sequence[int] = (3, 4, 5, 6)
    ga2m_interaction_grid: Sequence[int] = (0, 5, 10, 20, 40)
    # KernelSHAP cost is O(n_explain * nsamples * n_background) and nsamples
    # grows with d; left at the library default it dominates a d=166 cell.
    shap_nsamples: int = 256
    shap_max_d: int = 400           # above this, skip and say so
    # Permutation importance: repeats x d forward passes, once per model.
    perm_repeats: int = 3
    perm_sample: int = 2000

    # --- bookkeeping --------------------------------------------------------
    quick: bool = False
    out_dir: str = "runs"

    def to_quick(self) -> "RunConfig":
        """A smoke-test configuration: same code path, ~2 orders less compute."""
        import copy
        c = copy.deepcopy(self)
        c.quick = True
        c.seeds = (0, 1)
        c.epochs = 5
        c.n_query = 4000
        c.n_holdout_mu = 2000
        c.sobol_mc_base = 256
        c.shap_n_explain = 40
        c.shap_n_background = 20
        c.n_lambda = 8
        c.max_iter = 400
        c.screen_m = 12
        c.sdt_epochs = 80
        c.n_bootstrap = 100
        c.budget_match = False
        return c

    def to_rigorous(self) -> "RunConfig":
        """The headline protocol: 10 seeds, 50 epochs, and a grid that fits.

        Every setting here was chosen against `tools/preflight.py` calibrated
        on measured wall-clock, not by picking round numbers. The naive
        "turn everything up" version projects to ~330 GPU-hours, which no
        Colab session survives; this one lands near 25-35 h across a handful
        of sessions while giving up nothing that the analysis depends on.

        What is NOT reduced, and why:
          seeds = 10   the entire point. The paired Wilcoxon tests in
                       pact/stats.py lose power fast below ~8, and with them
                       the justification for a multi-seed run at all.
          epochs = 50  the stated protocol; teacher training is a few percent
                       of a cell, so this is nearly free.

        What is reduced, and why it is safe:
          sobol_mc_base 8192 -> 4096
              The reference noise floor is what N controls, and the shipped
              reduced run already measured floors of 0.007-0.055 at N=256,
              far below every Delta. N=4096 is a 16x margin on that. The
              records report the floor per cell, so this assumption is
              checkable rather than assumed -- if floor > Delta anywhere,
              raise N for that dataset and rerun those cells only.
          screen_m 32 -> 24
              FISTA is quadratic in |A|, and at d=166 this takes |A| from
              2153 to 1493 (a 2.1x saving on the dominant stage) while still
              allowing pairs among 24 of 166 features. The truncation floor is
              a reported failure mode either way; m is swept in the ablation.
          max_iter 2000 -> 800
              With warm starts down the lambda path this rarely binds. Each
              fit now records `converged`, so the assumption is verified in
              the output instead of trusted.
          n_lambda 20 -> 14
              Still a smooth frontier curve; the operating point is chosen by
              the knee rule, which does not need a dense path.
          n_query 20000 -> 12000
              The Gram is O(|Z| |A|^2) and built once per cell.
          budget_match stays ON but the grids are trimmed to three settings
              each; sweeping five costs more than the comparison is worth.
        """
        import copy
        c = copy.deepcopy(self)
        c.quick = False
        c.seeds = tuple(range(10))
        c.epochs = 50
        c.sobol_mc_base = 4096
        c.screen_m = 24
        c.max_iter = 800
        c.n_lambda = 14
        c.n_query = 12000
        c.n_holdout_mu = 8000
        c.n_bootstrap = 1000
        c.budget_match = True
        c.sdt_depth_grid = (4, 5, 6)
        c.ga2m_interaction_grid = (5, 20, 40)
        return c

    def as_dict(self) -> dict:
        d = asdict(self)
        d["seeds"] = list(self.seeds)
        d["hidden"] = list(self.hidden)
        return d


RHO_ABLATION = (0.0, 0.25, 0.5, 1.0)
TEACHER_ABLATION = ("mlp", "gbm")


### 1.2 Seeding, the product reference measure, and the certificate bound

In [ ]:
"""Seeding, device selection and small numerical helpers."""
from __future__ import annotations

import os
import random

import numpy as np

try:  # torch is optional at import time so the package is testable without it
    import torch
    HAS_TORCH = True
except Exception:  # pragma: no cover
    torch = None
    HAS_TORCH = False


def set_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    if HAS_TORCH:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)


def device() -> str:
    if HAS_TORCH and torch.cuda.is_available():
        return "cuda"
    return "cpu"


def centred_logits(log_proba: np.ndarray) -> np.ndarray:
    """Centre log-probabilities across classes.

    The softmax is invariant to a per-row constant, so the raw logits are only
    identified up to that constant.  Centring fixes the gauge, which matters
    because every L2 quantity in this project is computed in logit space.
    """
    lp = np.asarray(log_proba, dtype=np.float64)
    return lp - lp.mean(axis=1, keepdims=True)


def safe_log_proba(proba: np.ndarray, eps: float = 1e-7) -> np.ndarray:
    p = np.clip(np.asarray(proba, dtype=np.float64), eps, 1.0)
    p = p / p.sum(axis=1, keepdims=True)
    return np.log(p)


def multioutput_variance(Y: np.ndarray) -> float:
    """Total variance of a vector-valued function: sum of per-output variances.

    This is the sigma_T^2 that appears in the certificate; summing over outputs
    is what makes the multi-class Sobol' spectrum in Eq. (4) a single
    probability distribution rather than one per class.
    """
    Y = np.asarray(Y, dtype=np.float64)
    return float(((Y - Y.mean(axis=0, keepdims=True)) ** 2).sum(axis=1).mean())


def relative_l2(Y_true: np.ndarray, Y_pred: np.ndarray) -> float:
    """eta = ||f_T - f_S||_{L2} / sigma(f_T), both in multi-output form."""
    Y_true = np.asarray(Y_true, dtype=np.float64)
    Y_pred = np.asarray(Y_pred, dtype=np.float64)
    num = float(((Y_true - Y_pred) ** 2).sum(axis=1).mean())
    den = multioutput_variance(Y_true)
    if den <= 0:
        return float("nan")
    return float(np.sqrt(num / den))


def certificate_bound(eta: float) -> float:
    """Delta(eta) = 4 eta / (1 - eta) for eta < 1.

    Returns +inf once eta >= 1, but that is a sentinel, not a bound: Theorem 1
    assumes eta < 1 and makes no claim outside it.  Callers must branch on the
    precondition rather than treat inf as a very weak certificate -- every test
    of the form `measured <= inf` passes trivially.  See
    `metrics.certificate_report`, which returns `precondition_violated` and
    None (not True) for its hold tests in that case.
    """
    if not np.isfinite(eta) or eta >= 1.0:
        return float("inf")
    return float(4.0 * eta / (1.0 - eta))


def product_measure_sample(X: np.ndarray, n: int, rng: np.random.Generator) -> np.ndarray:
    """Draw from mu = prod_j mu_hat_j by resampling each column independently.

    The result has exactly the empirical marginals of X and independent
    coordinates -- i.e. it is the reference measure the ANOVA decomposition is
    orthogonal under.
    """
    X = np.asarray(X)
    n_rows = X.shape[0]
    out = np.empty((n, X.shape[1]), dtype=X.dtype)
    for j in range(X.shape[1]):
        out[:, j] = X[rng.integers(0, n_rows, size=n), j]
    return out


### 1.3 Orthonormal Legendre basis and the screened hyperbolic index set

In [ ]:
"""Orthonormal tensor-Legendre basis and the screened hyperbolic index set.

On [0,1]^d with the uniform measure, the shifted Legendre polynomials
    psi_n(z) = sqrt(2n+1) * P_n(2z - 1)
are orthonormal, and products Psi_alpha(z) = prod_j psi_{alpha_j}(z_j) form an
orthonormal basis of L^2.  Every basis function belongs to exactly one HDMR
component, identified by its support u(alpha) = {j : alpha_j > 0}, which is
what makes Eq. (2) of the report *be* the ANOVA decomposition rather than an
approximation of it.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np


def legendre_1d(z: np.ndarray, degree: int) -> np.ndarray:
    """Orthonormal shifted Legendre values. Returns (n, degree+1)."""
    z = np.asarray(z, dtype=np.float64).ravel()
    t = 2.0 * z - 1.0
    out = np.empty((z.size, degree + 1), dtype=np.float64)
    out[:, 0] = 1.0
    if degree >= 1:
        out[:, 1] = t
    for n in range(1, degree):
        out[:, n + 1] = ((2 * n + 1) * t * out[:, n] - n * out[:, n - 1]) / (n + 1)
    for n in range(degree + 1):
        out[:, n] *= np.sqrt(2.0 * n + 1.0)
    return out


@dataclass
class IndexSet:
    """A sparse multi-index set stored as a list of (feature, degree) pairs."""
    terms: list[tuple[tuple[int, ...], tuple[int, ...]]]  # (feats, degrees)
    d: int
    degree: int

    def __len__(self) -> int:
        return len(self.terms)

    @property
    def supports(self) -> list[tuple[int, ...]]:
        return [feats for feats, _ in self.terms]

    def groups(self) -> dict[tuple[int, ...], np.ndarray]:
        """Map each HDMR support u -> array of term positions. Excludes u=()."""
        g: dict[tuple[int, ...], list[int]] = {}
        for i, (feats, _) in enumerate(self.terms):
            if not feats:
                continue
            g.setdefault(feats, []).append(i)
        return {k: np.asarray(v, dtype=np.int64) for k, v in g.items()}

    def constant_position(self) -> int:
        for i, (feats, _) in enumerate(self.terms):
            if not feats:
                return i
        raise ValueError("index set has no constant term")


def _hyperbolic_ok(degrees: tuple[int, ...], p: int, q: float) -> bool:
    s = sum(float(a) ** q for a in degrees)
    return s ** (1.0 / q) <= p + 1e-12


def build_index_set(d: int, degree: int, q_norm: float,
                    pair_features: np.ndarray | None = None,
                    max_order: int = 2) -> IndexSet:
    """Pass-one (all univariate) plus pass-two (pairs among `pair_features`).

    Keeping every first-order term for all d features means no feature is ever
    unattributable at order 1; screening only limits which *interactions* are
    representable, which is the truncation whose cost the report reports.
    """
    terms: list[tuple[tuple[int, ...], tuple[int, ...]]] = [((), ())]

    for j in range(d):
        for a in range(1, degree + 1):
            if _hyperbolic_ok((a,), degree, q_norm):
                terms.append(((j,), (a,)))

    if max_order >= 2 and pair_features is not None and len(pair_features) >= 2:
        feats = sorted(int(f) for f in pair_features)
        for ii in range(len(feats)):
            for jj in range(ii + 1, len(feats)):
                i, j = feats[ii], feats[jj]
                for a in range(1, degree + 1):
                    for b in range(1, degree + 1):
                        if _hyperbolic_ok((a, b), degree, q_norm):
                            terms.append(((i, j), (a, b)))
    return IndexSet(terms=terms, d=d, degree=degree)


def design_matrix(Z: np.ndarray, index_set: IndexSet,
                  chunk: int = 20000) -> np.ndarray:
    """Evaluate all basis functions. Returns (n, P) float64.

    Univariate factors are computed once per feature and reused, so the cost is
    O(n * d * p) for the factors plus O(n * P) for the products.
    """
    Z = np.asarray(Z, dtype=np.float64)
    n, d = Z.shape
    P = len(index_set)
    out = np.empty((n, P), dtype=np.float64)

    for start in range(0, n, chunk):
        stop = min(start + chunk, n)
        blk = Z[start:stop]
        # factors[j] is (chunk, degree+1)
        factors = [legendre_1d(blk[:, j], index_set.degree) for j in range(d)]
        for t, (feats, degs) in enumerate(index_set.terms):
            if not feats:
                out[start:stop, t] = 1.0
            elif len(feats) == 1:
                out[start:stop, t] = factors[feats[0]][:, degs[0]]
            else:
                v = factors[feats[0]][:, degs[0]]
                for f, a in zip(feats[1:], degs[1:]):
                    v = v * factors[f][:, a]
                out[start:stop, t] = v
    return out


### 1.4 GAS-PCE — the group-sparse HDMR student (FISTA + analytic Sobol')

In [ ]:
"""GAS-PCE: group-sparse HDMR polynomial chaos student.

Solves

    min_c  (1/N) || Psi c - Y ||_F^2  +  lambda * sum_u w_u || c_{A_u,:} ||_F
    w_u = sqrt(|A_u|) * gamma^(|u|-1)

by FISTA with blockwise soft-thresholding.  The gradient is formed from the
precomputed Gram matrix G = Psi^T Psi / N and B = Psi^T Y / N, so each
iteration costs one P x P matvec rather than a pass over the N x P design --
this is what makes a 20-value lambda path with 2000 iterations cheap enough to
run 10 seeds x 4 datasets in a Colab session.

Sobol' indices come straight off the coefficients (Eq. 4): with an orthonormal
basis the variance of HDMR component u is the squared Frobenius norm of its
coefficient block.  No sampling, no estimator variance.
"""
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np


# ---------------------------------------------------------------------------
# solver
# ---------------------------------------------------------------------------


def _group_prox(C: np.ndarray, groups: list[np.ndarray], thresh: np.ndarray,
                skip: int) -> np.ndarray:
    """Blockwise soft-threshold; the constant term (row `skip`) is untouched."""
    out = C.copy()
    for g, t in zip(groups, thresh):
        block = out[g]
        nrm = np.sqrt((block ** 2).sum())
        if nrm <= t or nrm == 0.0:
            out[g] = 0.0
        else:
            out[g] = block * (1.0 - t / nrm)
    return out


def fista_group_lasso(G: np.ndarray, B: np.ndarray, groups: list[np.ndarray],
                      weights: np.ndarray, lam: float, const_pos: int,
                      C0: np.ndarray | None = None, max_iter: int = 2000,
                      tol: float = 1e-7, L: float | None = None):
    """Minimise (1/N)||Psi C - Y||^2 + lam * sum_u w_u ||C_u||_F via FISTA."""
    P, K = B.shape
    C = np.zeros((P, K)) if C0 is None else C0.copy()
    Zk = C.copy()
    t = 1.0
    converged = False

    if L is None:
        L = 2.0 * _power_iteration(G)
    step = 1.0 / max(L, 1e-12)
    thresh = step * lam * weights

    prev_obj = np.inf
    obj = np.inf
    n_iter = 0
    for it in range(max_iter):
        n_iter = it + 1
        grad = 2.0 * (G @ Zk - B)
        C_new = _group_prox(Zk - step * grad, groups, thresh, const_pos)
        t_new = 0.5 * (1.0 + np.sqrt(1.0 + 4.0 * t * t))
        Zk = C_new + ((t - 1.0) / t_new) * (C_new - C)
        C, t = C_new, t_new

        if it % 25 == 0 or it == max_iter - 1:
            obj = _objective(G, B, C, groups, weights, lam)
            # A non-finite objective means the iterate has diverged (step too
            # large for an ill-conditioned Gram, or an overflow in the design).
            # Reporting it as "not converged" understates the problem: the
            # coefficients are unusable, not merely imprecise.
            if not np.isfinite(obj):
                return C, False, {"n_iter": n_iter, "objective": float("nan"),
                                  "status": "diverged"}
            if np.isfinite(prev_obj) and abs(prev_obj - obj) <= tol * max(1.0, abs(prev_obj)):
                converged = True
                break
            prev_obj = obj
    status = "converged" if converged else "max_iter"
    return C, converged, {"n_iter": n_iter, "objective": float(obj),
                          "status": status}


def gram_conditioning(G: np.ndarray, max_p: int = 2600) -> dict:
    """Rank, condition number and eigenvalue extremes of the Gram matrix.

    G = Psi'Psi / n is symmetric positive semi-definite, so `eigvalsh` gives
    the singular values directly and is both faster and more accurate here
    than a general SVD. Cost is O(P^3) once per cell, which is a small
    fraction of a path fit, but it is capped: above `max_p` the decomposition
    would cost more than the fit it is diagnosing, so the fields come back
    None rather than silently costing minutes.

    Why this is recorded. A rank-deficient or badly conditioned design is the
    mechanism by which a *correct* solver returns coefficients that move
    between environments (different BLAS orderings resolve a near-null space
    differently). Without these numbers, that failure mode is indistinguishable
    from a genuine modelling result -- so if two runs ever disagree while the
    provenance fingerprints match, this is the first field to read.

    `rank` counts eigenvalues above `eps * P * lambda_max`, the standard
    numerical-rank convention (numpy.linalg.matrix_rank).
    """
    P = int(G.shape[0])
    out: dict = {"n_terms": P, "rank": None, "cond": None,
                 "eig_max": None, "eig_min": None,
                 "rank_deficient": None, "skipped": False}
    if P == 0:
        return out
    if P > max_p:
        out["skipped"] = True
        out["skip_reason"] = f"P={P} exceeds max_p={max_p}"
        # the Lipschitz constant is computed anyway, so report the cheap half
        out["eig_max"] = float(_power_iteration(G))
        return out
    w = np.linalg.eigvalsh(np.asarray(G, dtype=np.float64))
    w = np.clip(w, 0.0, None)            # PSD up to round-off
    emax = float(w[-1])
    tol = float(np.finfo(np.float64).eps) * P * emax
    rank = int((w > tol).sum())
    # smallest eigenvalue retained as *signal*; the condition number is
    # reported over that retained subspace, which is what the solver sees
    emin = float(w[w > tol].min()) if rank else 0.0
    out.update({
        "rank": rank,
        "eig_max": emax,
        "eig_min": emin,
        "cond": float(emax / emin) if emin > 0 else float("inf"),
        "rank_deficient": bool(rank < P),
        "rank_tol": tol,
    })
    return out


def prune_for_conditioning(G: np.ndarray, max_cond: float = 1e10,
                           protect: int | None = None) -> np.ndarray:
    """Basis columns to drop so the retained Gram has cond <= `max_cond`.

    Why dropping beats stabilising. On gas-drift at rho=0 the design comes out
    rank 1340 of 1341 with the retained spectrum still spanning 2.6e12, and
    its smallest retained eigenvalue sits a factor of 1.3 above the
    machine-epsilon rank threshold -- i.e. several directions are effectively
    unidentified and the eps cutoff keeps one of them by luck. Coefficients
    along such a direction are not determined by the data, so whatever the
    solver returns there is arbitrary, and the Sobol' index read off it is
    arbitrary too.

    Two ways to make that reproducible, and only one works:

      * a diagonal ridge makes the objective strictly convex, so the minimiser
        is unique -- but the stabilised direction then has curvature 2*eps and
        FISTA needs O(sqrt(1/eps)) iterations to converge along it. At
        eps=1e-8 that is ~1e4 iterations against the 800 a production run
        uses, so the solution is unique in theory and still start-dependent in
        practice. Measured: start-dependence fell only 12x. Not a fix.
      * dropping the offending columns removes the direction entirely. The
        retained problem is well conditioned, FISTA converges at its normal
        rate, nothing is biased in the subspace that survives, and the dropped
        terms get a Sobol' index of exactly zero -- which is the honest
        reading, since the data does not identify them.

    So: truncated-spectrum selection. Eigen-decompose G, take every direction
    with lambda < lambda_max / max_cond, and for each one drop the basis column
    carrying the largest loading in it (skipping already-dropped columns and
    the constant term, which must survive to represent the mean).

    Returns the column indices to drop, ascending.

    The pruning ITERATES. Dropping one column changes the spectrum of what
    remains and can expose a direction that was previously masked, so a single
    pass is not enough: on a design with an exactly duplicated feature, one
    pass left the retained Gram still rank deficient (25 of 26). Each round
    re-decomposes the surviving submatrix and stops when no direction falls
    below the threshold, or when nothing further may be dropped. Cost is a few
    `eigh` calls at O(P^3), the same order already paid by
    `gram_conditioning`; `max_rounds` bounds it.
    """
    P = int(G.shape[0])
    if P == 0 or not np.isfinite(max_cond) or max_cond <= 1.0:
        return np.empty(0, dtype=int)
    G = np.asarray(G, dtype=np.float64)
    alive = np.arange(P)                   # original indices still in play
    dropped: list[int] = []
    max_rounds = 12
    for _ in range(max_rounds):
        sub = G[np.ix_(alive, alive)]
        w, V = np.linalg.eigh(sub)
        emax = float(np.clip(w, 0.0, None).max())
        if emax <= 0.0:
            break
        # Threshold on BOTH the target condition number and the numerical rank
        # cutoff: the second is what makes "rank deficient" true, and leaving
        # it unsatisfied would report a design the solver still cannot pin.
        thresh = max(emax / float(max_cond),
                     float(np.finfo(np.float64).eps) * sub.shape[0] * emax)
        bad = np.where(w < thresh)[0]
        if bad.size == 0:
            break
        round_drop: list[int] = []
        for k in bad:
            loading = np.abs(V[:, k])
            if round_drop:
                loading[np.asarray(round_drop, dtype=int)] = -1.0
            if protect is not None:
                prot = np.where(alive == protect)[0]
                if prot.size:
                    loading[int(prot[0])] = -1.0
            j = int(np.argmax(loading))
            if loading[j] <= 0.0:
                continue
            round_drop.append(j)
        if not round_drop:
            break
        dropped.extend(int(alive[j]) for j in round_drop)
        alive = np.delete(alive, np.asarray(sorted(round_drop), dtype=int))
        if alive.size <= 1:
            break
    return np.array(sorted(set(dropped)), dtype=int)


def _power_iteration(G: np.ndarray, n_iter: int = 60) -> float:
    rng = np.random.default_rng(0)
    v = rng.standard_normal(G.shape[0])
    v /= np.linalg.norm(v)
    lam = 0.0
    for _ in range(n_iter):
        w = G @ v
        nw = np.linalg.norm(w)
        if nw == 0:
            return 0.0
        v = w / nw
        lam = nw
    return float(lam)


def _objective(G, B, C, groups, weights, lam) -> float:
    # (1/N)||Psi C - Y||^2 = C'GC - 2C'B + const; the constant is dropped.
    quad = float(np.sum(C * (G @ C)) - 2.0 * np.sum(C * B))
    pen = float(lam * sum(w * np.sqrt((C[g] ** 2).sum())
                          for g, w in zip(groups, weights)))
    return quad + pen


# ---------------------------------------------------------------------------
# model
# ---------------------------------------------------------------------------


@dataclass
class PCEFit:
    """One point on the lambda path."""
    lam: float
    coef: np.ndarray                       # (P, K)
    active_components: int
    nnz: int
    max_order: int
    sobol: dict[tuple[int, ...], float] = field(default_factory=dict)
    total_effects: np.ndarray | None = None
    converged: bool = True
    # solver diagnostics for this lambda, so a suspect number can be traced to
    # the solve that produced it rather than re-derived from the edit history
    n_iter: int = 0
    objective: float = float("nan")
    status: str = "unknown"
    train_mse: float = float("nan")        # (1/n)||Psi C - Y||^2 on the query set


class GasPCE:
    """Group-sparse HDMR--PCE student."""

    def __init__(self, d: int, degree: int = 4, q_norm: float = 0.9,
                 screen_m: int = 32, gamma: float = 2.0, max_order: int = 2,
                 n_lambda: int = 20, lambda_min_ratio: float = 1e-4,
                 max_iter: int = 2000, tol: float = 1e-7,
                 cond_max_p: int = 2600, max_cond: float = 1e10):
        self.d = d
        self.degree = degree
        self.q_norm = q_norm
        self.screen_m = screen_m
        self.gamma = gamma
        self.max_order = max_order
        self.n_lambda = n_lambda
        self.lambda_min_ratio = lambda_min_ratio
        self.max_iter = max_iter
        self.tol = tol
        self.cond_max_p = cond_max_p
        self.max_cond = max_cond

        self.index_set: IndexSet | None = None
        self.path: list[PCEFit] = []
        self.screened_features_: np.ndarray | None = None
        self.all_converged_: bool = True
        self.n_not_converged_: int = 0
        self.conditioning_: dict = {}
        self.pruned_terms_: np.ndarray = np.empty(0, dtype=int)
        self._keep_cols_ = None
        self._n_cols_full_: int = 0

    # -- internals ---------------------------------------------------------

    def _groups_and_weights(self, index_set: IndexSet):
        gmap = index_set.groups()
        supports = list(gmap.keys())
        groups = [gmap[u] for u in supports]
        weights = np.array(
            [np.sqrt(len(gmap[u])) * (self.gamma ** (len(u) - 1))
             for u in supports], dtype=np.float64)
        return supports, groups, weights

    @staticmethod
    def _lambda_max(B: np.ndarray, groups, weights) -> float:
        vals = [np.sqrt((B[g] ** 2).sum()) / max(w, 1e-12)
                for g, w in zip(groups, weights)]
        return float(2.0 * max(vals)) if vals else 1.0

    def _screen(self, Z: np.ndarray, Y: np.ndarray) -> np.ndarray:
        """Pass one: fit first-order only, rank features by explained variance."""
        iset = build_index_set(self.d, self.degree, self.q_norm,
                               pair_features=None, max_order=1)
        Psi = design_matrix(Z, iset)
        n = Psi.shape[0]
        G = (Psi.T @ Psi) / n
        B = (Psi.T @ Y) / n
        supports, groups, weights = self._groups_and_weights(iset)
        # unpenalised least squares with a touch of ridge: screening only needs
        # a ranking, not a sparse model
        C = np.linalg.solve(G + 1e-6 * np.eye(G.shape[0]), B)
        energy = np.array([float((C[g] ** 2).sum()) for g in groups])
        order = np.argsort(-energy)
        feats = [supports[i][0] for i in order]
        m = min(self.screen_m, self.d)
        return np.asarray(feats[:m], dtype=np.int64)

    # -- public API --------------------------------------------------------

    def fit(self, Z: np.ndarray, Y: np.ndarray) -> "GasPCE":
        """Fit the whole lambda path. Z in [0,1]^d, Y the teacher logits."""
        Z = np.asarray(Z, dtype=np.float64)
        Y = np.asarray(Y, dtype=np.float64)
        if Y.ndim == 1:
            Y = Y[:, None]

        self.screened_features_ = self._screen(Z, Y)
        self.index_set = build_index_set(
            self.d, self.degree, self.q_norm,
            pair_features=self.screened_features_, max_order=self.max_order)

        Psi = design_matrix(Z, self.index_set)
        n = Psi.shape[0]
        G = (Psi.T @ Psi) / n
        B = (Psi.T @ Y) / n
        del Psi

        supports, groups, weights = self._groups_and_weights(self.index_set)
        const_pos = self.index_set.constant_position()

        # Conditioning of the design, recorded before any solve. If two runs
        # ever disagree with matching provenance, this is the field to read:
        # a near-null space is how a correct solver returns environment-
        # dependent coefficients.
        self.conditioning_ = gram_conditioning(G, max_p=self.cond_max_p)
        eig_max = float(self.conditioning_.get("eig_max")
                        or _power_iteration(G))

        # Drop basis columns the data does not identify, so the retained
        # problem is well conditioned and its minimiser unique. See
        # `prune_for_conditioning` for why this is done instead of a ridge:
        # a ridge makes the minimiser unique but leaves FISTA unable to reach
        # it in a production iteration budget, which is not reproducibility.
        n_cols_full = int(G.shape[0])
        self.pruned_terms_ = np.empty(0, dtype=int)
        keep = None
        # full-column-space copy of the groups, kept in lockstep with
        # `supports`; identical to `groups` unless columns are pruned
        groups_full = list(groups)
        if self.max_cond and np.isfinite(self.max_cond):
            drop = prune_for_conditioning(G, max_cond=self.max_cond,
                                          protect=const_pos)
            if drop.size:
                self.pruned_terms_ = drop
                keep = np.setdiff1d(np.arange(n_cols_full), drop)
                # Column j of the full design becomes column pos[j] of the
                # reduced one, or -1 if dropped.
                pos = np.full(n_cols_full, -1, dtype=int)
                pos[keep] = np.arange(keep.size)
                # Remap the groups and filter supports/weights IN LOCKSTEP: a
                # group that loses every column must disappear from all three
                # lists together, or the penalty weights and the Sobol'
                # supports silently misalign with the coefficient blocks.
                kept_groups, kept_full, kept_supports, kept_weights = [], [], [], []
                for g, u, w in zip(groups, supports, weights):
                    g_keep = g[pos[g] >= 0]          # full-space indices kept
                    g2 = pos[g_keep]                 # reduced-space indices
                    if g2.size:
                        kept_groups.append(g2)
                        kept_full.append(g_keep)
                        kept_supports.append(u)
                        kept_weights.append(w)
                groups, supports = kept_groups, kept_supports
                groups_full = kept_full
                weights = np.asarray(kept_weights, dtype=np.float64)
                G = G[np.ix_(keep, keep)]
                B = B[keep]
                const_pos = (int(pos[const_pos])
                             if const_pos is not None and pos[const_pos] >= 0
                             else None)
                after = gram_conditioning(G, max_p=self.cond_max_p)
                self.conditioning_.update({
                    "pruned": int(drop.size),
                    "post_prune_n_terms": after["n_terms"],
                    "post_prune_rank": after["rank"],
                    "post_prune_cond": after["cond"],
                    "post_prune_rank_deficient": after["rank_deficient"],
                })
                eig_max = float(after.get("eig_max") or _power_iteration(G))

        self._keep_cols_ = keep
        self._n_cols_full_ = n_cols_full
        L = 2.0 * eig_max

        # ||Y||^2 per sample, the reference for the training residual below
        yy = float((Y ** 2).sum(axis=1).mean())

        lam_max = self._lambda_max(B, groups, weights)
        lams = np.geomspace(lam_max, lam_max * self.lambda_min_ratio,
                            self.n_lambda)

        self.path = []
        self.n_not_converged_ = 0
        C = None
        for lam in lams:
            C, ok, diag = fista_group_lasso(
                G, B, groups, weights, float(lam), const_pos, C0=C,
                max_iter=self.max_iter, tol=self.tol, L=L)
            if not ok:
                self.n_not_converged_ += 1
            # Scatter back into full column space before summarising: the
            # design matrices built by `design_for` carry every term of the
            # index set, so a reduced coefficient vector would silently
            # misalign with them. Pruned terms get exact zeros, which is also
            # the right Sobol' reading -- the data does not identify them.
            C_full = C
            if self._keep_cols_ is not None:
                C_full = np.zeros((self._n_cols_full_, C.shape[1]),
                                  dtype=C.dtype)
                C_full[self._keep_cols_] = C
            fit = self._summarise(C_full, lam, supports, groups_full)
            fit.converged = ok
            fit.n_iter = int(diag["n_iter"])
            fit.objective = float(diag["objective"])
            fit.status = str(diag["status"])
            # (1/n)||Psi C - Y||^2 = C'GC - 2C'B + (1/n)||Y||^2, computed from
            # the Gram so the design matrix need not be kept alive.
            fit.train_mse = float(
                np.sum(C * (G @ C)) - 2.0 * np.sum(C * B) + yy)
            self.path.append(fit)
        self.all_converged_ = self.n_not_converged_ == 0
        return self

    def _summarise(self, C, lam, supports, groups) -> PCEFit:
        energies = {}
        for u, g in zip(supports, groups):
            e = float((C[g] ** 2).sum())
            if e > 0:
                energies[u] = e
        total = sum(energies.values())
        sobol = {u: e / total for u, e in energies.items()} if total > 0 else {}
        te = np.zeros(self.d)
        for u, s in sobol.items():
            for j in u:
                te[j] += s
        return PCEFit(
            lam=float(lam),
            coef=C.copy(),
            active_components=len(energies),
            nnz=int((np.abs(C) > 0).sum()),
            max_order=max((len(u) for u in energies), default=0),
            sobol=sobol,
            total_effects=te,
        )

    def predict_logits(self, Z: np.ndarray, fit: PCEFit) -> np.ndarray:
        """Convenience path: builds the design matrix every call.

        Do NOT use this inside a loop over the lambda path -- building Psi is
        the dominant cost at d>100 and the matrix is identical for every fit.
        Use `design_for` once and `predict_from_design` per fit instead.
        """
        assert self.index_set is not None
        return self.predict_from_design(self.design_for(Z), fit)

    def design_for(self, Z: np.ndarray) -> np.ndarray:
        """Build (and let the caller keep) the design matrix for one sample."""
        assert self.index_set is not None
        return design_matrix(np.asarray(Z, dtype=np.float64), self.index_set)

    @staticmethod
    def predict_from_design(Psi: np.ndarray, fit: PCEFit) -> np.ndarray:
        return Psi @ fit.coef


### 1.5 Dataset loading and preprocessing

In [ ]:
"""Dataset loading and preprocessing.

Preprocessing decisions that affect the explanation are deliberate and are
documented in the report:

* Categorical columns are **ordinal-encoded by category frequency**, not
  one-hot encoded, so that one semantic feature maps to exactly one Sobol'
  index.  One-hot encoding would split a single feature's variance across many
  indices and make the HDMR component functions uninterpretable.
* Numeric columns pass through untouched before the quantile map.
* The quantile transform is fitted on the training split only and maps each
  marginal to U[0,1], which is what makes the shifted Legendre basis
  orthonormal under the reference product measure.
"""
from __future__ import annotations

import time
from dataclasses import dataclass
from functools import lru_cache

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer


@dataclass
class Dataset:
    key: str
    X_train: np.ndarray      # quantile-transformed, in [0, 1]
    X_val: np.ndarray
    X_test: np.ndarray
    y_train: np.ndarray
    y_val: np.ndarray
    y_test: np.ndarray
    feature_names: list[str]
    n_classes: int

    @property
    def d(self) -> int:
        return self.X_train.shape[1]

    def summary(self) -> dict:
        return {
            "key": self.key,
            "n_train": int(self.X_train.shape[0]),
            "n_val": int(self.X_val.shape[0]),
            "n_test": int(self.X_test.shape[0]),
            "d": int(self.d),
            "n_classes": int(self.n_classes),
        }


@lru_cache(maxsize=8)
def _fetch_cached(openml_id: int, openml_name: str, key: str):
    """Fetch from OpenML by id, falling back to the name if the id moves.

    Memoised for the life of the process. Without this the grid refetches and
    re-parses the same ARFF once per cell -- 40 times per dataset over a
    4-rho, 10-seed ablation. sklearn's on-disk cache does not save you: it is
    ephemeral on Colab, re-parsing the frame still costs seconds at d=166, and
    a corrupted entry makes every cell redownload (the "Invalid cache,
    redownloading file" warning) which turns one transient network error into
    a failed cell.

    Transient fetch errors are retried before the name fallback, because a
    single dropped connection 90 cells into a grid should not cost the cell.
    """
    from sklearn.datasets import fetch_openml

    last = None
    for attempt in range(3):
        try:
            return fetch_openml(data_id=openml_id, as_frame=True,
                                parser="auto")
        except Exception as exc:  # pragma: no cover - network dependent
            last = exc
            if attempt < 2:
                time.sleep(2.0 * (attempt + 1))
    try:
        return fetch_openml(name=openml_name, version=1, as_frame=True,
                            parser="auto")
    except Exception as exc_name:
        raise RuntimeError(
            f"could not fetch {key}: id lookup failed after 3 attempts "
            f"({last}); name lookup failed ({exc_name})"
        ) from exc_name


def _fetch(spec: DatasetSpec):
    return _fetch_cached(spec.openml_id, spec.openml_name, spec.key)


def _encode_frame(df: pd.DataFrame) -> tuple[np.ndarray, list[str]]:
    """Ordinal-encode categoricals by frequency; median-impute numerics."""
    cols, names = [], []
    for name in df.columns:
        s = df[name]
        if pd.api.types.is_numeric_dtype(s):
            v = pd.to_numeric(s, errors="coerce").astype(float)
            v = v.fillna(v.median() if np.isfinite(v.median()) else 0.0)
            cols.append(v.to_numpy())
        else:
            s = s.astype("object").where(s.notna(), "__missing__")
            order = s.value_counts().index.tolist()
            mapping = {cat: i for i, cat in enumerate(order)}
            cols.append(s.map(mapping).astype(float).to_numpy())
        names.append(str(name))
    return np.column_stack(cols), names


def load_dataset(spec: DatasetSpec, seed: int, test_size: float = 0.25,
                 val_size: float = 0.10) -> Dataset:
    """Load, encode and split one dataset. Memoised on (key, seed, splits).

    The grid visits the same (dataset, seed) once per rho, and the split and
    quantile transform are deterministic in those arguments, so recomputing
    them per rho is pure waste -- the transform alone is a sort over the
    training split. The cache holds two datasets, which is what the
    seed-major loop order in `experiment.run_grid` needs; `DatasetSpec` is a
    frozen dataclass so it hashes by value and a config change produces a
    different key rather than a stale hit.
    """
    return _load_dataset_cached(spec, int(seed), float(test_size),
                                float(val_size))


@lru_cache(maxsize=2)
def _load_dataset_cached(spec: DatasetSpec, seed: int, test_size: float,
                         val_size: float) -> Dataset:
    bunch = _fetch(spec)
    X_df = bunch.data
    if not isinstance(X_df, pd.DataFrame):
        X_df = pd.DataFrame(X_df)

    y_raw = pd.Series(np.asarray(bunch.target).ravel())
    classes = sorted(y_raw.astype(str).unique())
    y = y_raw.astype(str).map({c: i for i, c in enumerate(classes)}).to_numpy()

    X, names = _encode_frame(X_df)

    # drop constant columns: they contribute no variance and would make the
    # quantile transform degenerate
    keep = X.std(axis=0) > 0
    X, names = X[:, keep], [n for n, k in zip(names, keep) if k]

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=seed, stratify=y)
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_tr, y_tr, test_size=val_size, random_state=seed, stratify=y_tr)

    qt = QuantileTransformer(
        output_distribution="uniform",
        n_quantiles=int(min(1000, max(10, X_tr.shape[0]))),
        subsample=200_000,
        random_state=seed,
    ).fit(X_tr)

    # clip strictly inside [0,1]: the Legendre recurrence is fine at the
    # endpoints but downstream log/odds computations are happier away from them
    def tf(A):
        return np.clip(qt.transform(A), 1e-6, 1 - 1e-6).astype(np.float64)

    return Dataset(
        key=spec.key,
        X_train=tf(X_tr), X_val=tf(X_va), X_test=tf(X_te),
        y_train=y_tr.astype(int), y_val=y_va.astype(int), y_test=y_te.astype(int),
        feature_names=names,
        n_classes=len(classes),
    )


### 1.6 Teachers: MLP (50 epochs) and gradient boosting, frozen after fit

In [ ]:
"""Teacher models. Trained, then frozen before any student sees them.

Primary teacher: a 3-hidden-layer MLP trained for exactly `epochs` epochs
(50 in the headline protocol).  Secondary teacher for the ablation: histogram
gradient boosting, included because the capacity argument for deep models on
tabular data is contested (Borisov et al., 2024) and the choice of teacher
should therefore be an experimental variable.

Every teacher exposes the same interface:
    .predict_logits(X) -> (n, K) centred log-probabilities
    .predict_proba(X)  -> (n, K)
"""
from __future__ import annotations

import numpy as np


if HAS_TORCH:
    import torch
    import torch.nn as nn


class BaseTeacher:
    n_classes: int

    def predict_proba(self, X: np.ndarray) -> np.ndarray:  # pragma: no cover
        raise NotImplementedError

    def predict_logits(self, X: np.ndarray) -> np.ndarray:
        return centred_logits(safe_log_proba(self.predict_proba(X)))

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.predict_proba(X).argmax(axis=1)


# ---------------------------------------------------------------------------
# MLP
# ---------------------------------------------------------------------------


class MLPTeacher(BaseTeacher):
    def __init__(self, d: int, n_classes: int, hidden=(256, 128, 64),
                 dropout: float = 0.1, lr: float = 1e-3,
                 weight_decay: float = 1e-4, epochs: int = 50,
                 batch_size: int = 256, seed: int = 0):
        if not HAS_TORCH:
            raise RuntimeError("MLPTeacher requires torch; use teacher='gbm'")
        self.n_classes = n_classes
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.weight_decay = weight_decay
        self.device = device()
        torch.manual_seed(seed)

        layers, prev = [], d
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers).to(self.device)
        self.history: list[dict] = []

    def fit(self, X: np.ndarray, y: np.ndarray,
            X_val: np.ndarray | None = None, y_val: np.ndarray | None = None):
        Xt = torch.as_tensor(X, dtype=torch.float32, device=self.device)
        yt = torch.as_tensor(y, dtype=torch.long, device=self.device)

        counts = np.bincount(y, minlength=self.n_classes).astype(np.float64)
        w = np.where(counts > 0, counts.sum() / np.maximum(counts, 1), 0.0)
        w = w / w[w > 0].mean()
        weight = torch.as_tensor(w, dtype=torch.float32, device=self.device)

        opt = torch.optim.AdamW(self.net.parameters(), lr=self.lr,
                                weight_decay=self.weight_decay)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.epochs)
        lossf = nn.CrossEntropyLoss(weight=weight)
        n = Xt.shape[0]

        for epoch in range(self.epochs):
            self.net.train()
            perm = torch.randperm(n, device=self.device)
            tot = 0.0
            for s in range(0, n, self.batch_size):
                idx = perm[s:s + self.batch_size]
                opt.zero_grad(set_to_none=True)
                loss = lossf(self.net(Xt[idx]), yt[idx])
                loss.backward()
                opt.step()
                tot += loss.detach().item() * idx.numel()
            sched.step()
            rec = {"epoch": epoch, "train_loss": tot / n}
            if X_val is not None:
                rec["val_acc"] = float((self.predict(X_val) == y_val).mean())
            self.history.append(rec)

        self.net.eval()
        for p in self.net.parameters():      # freeze
            p.requires_grad_(False)
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        self.net.eval()
        out = []
        with torch.no_grad():
            Xt = torch.as_tensor(np.asarray(X), dtype=torch.float32,
                                 device=self.device)
            for s in range(0, Xt.shape[0], 4096):
                logits = self.net(Xt[s:s + 4096])
                out.append(torch.softmax(logits, dim=1).cpu().numpy())
        return np.concatenate(out, axis=0).astype(np.float64)


# ---------------------------------------------------------------------------
# Gradient boosting
# ---------------------------------------------------------------------------


class GBMTeacher(BaseTeacher):
    def __init__(self, n_classes: int, seed: int = 0, max_iter: int = 200):
        from sklearn.ensemble import HistGradientBoostingClassifier
        self.n_classes = n_classes
        self.model = HistGradientBoostingClassifier(
            max_iter=max_iter, random_state=seed, early_stopping=False)

    def fit(self, X, y, X_val=None, y_val=None):
        self.model.fit(X, y)
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return np.asarray(self.model.predict_proba(X), dtype=np.float64)


class SklearnMLPTeacher(BaseTeacher):
    """CPU fallback used only when torch is unavailable (e.g. local smoke tests)."""

    def __init__(self, n_classes: int, hidden=(256, 128, 64), epochs: int = 50,
                 seed: int = 0, lr: float = 1e-3):
        from sklearn.neural_network import MLPClassifier
        self.n_classes = n_classes
        self.model = MLPClassifier(hidden_layer_sizes=tuple(hidden),
                                   max_iter=epochs, learning_rate_init=lr,
                                   random_state=seed)

    def fit(self, X, y, X_val=None, y_val=None):
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            self.model.fit(X, y)
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return np.asarray(self.model.predict_proba(X), dtype=np.float64)


def build_teacher(kind: str, d: int, n_classes: int, cfg, seed: int):
    if kind == "gbm":
        return GBMTeacher(n_classes=n_classes, seed=seed)
    if kind == "mlp":
        if HAS_TORCH:
            return MLPTeacher(d=d, n_classes=n_classes, hidden=cfg.hidden,
                              dropout=cfg.dropout, lr=cfg.lr,
                              weight_decay=cfg.weight_decay, epochs=cfg.epochs,
                              batch_size=cfg.batch_size, seed=seed)
        return SklearnMLPTeacher(n_classes=n_classes, hidden=cfg.hidden,
                                 epochs=cfg.epochs, seed=seed, lr=cfg.lr)
    raise ValueError(f"unknown teacher kind: {kind}")


### 1.7 The two literature baselines: soft decision tree and GA2M

In [ ]:
"""The two interpretable-student baselines taken from the literature.

SDT  -- soft decision tree (Frosst & Hinton, 2017).  Chosen because it is the
        canonical interpretable student and because its inductive bias is
        maximally unlike an additive decomposition: hierarchical, axis-mixing
        at the gates, and non-orthogonal.  It isolates what hierarchy costs.

GA2M -- additive model with pairwise interactions (Lou et al., 2013),
        distilled on teacher logits exactly as in Distill-and-Compare
        (Tan et al., 2018).  Chosen because it is the closest functional
        analogue to a degree-limited HDMR -- same interaction order, same
        additive reading -- but with a non-orthogonal basis, so comparing it
        with GAS-PCE isolates the contribution of *orthogonality* rather than
        of additive structure.

Both regress the teacher's centred logits, so all three students are fitted to
the same target and their residuals are directly comparable.
"""
from __future__ import annotations

import numpy as np


if HAS_TORCH:
    import torch
    import torch.nn as nn


# ---------------------------------------------------------------------------
# Soft decision tree
# ---------------------------------------------------------------------------


class SoftDecisionTree:
    """Full binary tree of depth `depth` with sigmoid gates and leaf vectors.

    Each internal node i routes with p_i(x) = sigmoid(beta * (w_i . x + b_i));
    a leaf's weight is the product of gate probabilities along its path.  The
    output is the probability-weighted sum of leaf logit vectors.
    """

    def __init__(self, d: int, n_out: int, depth: int = 5, beta: float = 4.0,
                 lr: float = 2e-2, epochs: int = 200, batch_size: int = 512,
                 weight_decay: float = 1e-5, seed: int = 0,
                 w_scale: float = 1.0, learn_beta: bool = True):
        if not HAS_TORCH:
            raise RuntimeError("SoftDecisionTree requires torch")
        torch.manual_seed(seed)
        self.depth = depth
        self.n_internal = 2 ** depth - 1
        self.n_leaves = 2 ** depth
        self.device = device()
        self.epochs = epochs
        self.batch_size = batch_size

        # Initialisation matters more than anything else here. With small
        # gate weights every sigmoid sits at 0.5, all 2^depth leaves receive
        # near-equal weight, and the tree emits a single constant -- the
        # classic soft-tree cold start, which parks the model at eta = 1.0
        # (exactly the constant-mean predictor) and never escapes. So: scale
        # the gate weights up, centre each split at the middle of the input
        # range (b = -sum(w)/2 for inputs on [0,1]), sharpen with beta, and
        # break leaf symmetry with a small random init.
        W0 = torch.randn(self.n_internal, d, device=self.device) * w_scale
        self.W = nn.Parameter(W0)
        self.b = nn.Parameter(-W0.sum(dim=1) / 2.0)
        self.leaf = nn.Parameter(
            torch.randn(self.n_leaves, n_out, device=self.device) * 0.01)
        self.log_beta = nn.Parameter(
            torch.tensor(float(np.log(beta)), device=self.device),
            requires_grad=bool(learn_beta))
        self.params = [self.W, self.b, self.leaf, self.log_beta]
        self.lr, self.weight_decay = lr, weight_decay

    @property
    def beta(self):
        return torch.exp(self.log_beta)

    def _leaf_weights(self, X):
        gate = torch.sigmoid(self.beta * (X @ self.W.T + self.b))  # (n, I)
        w = torch.ones(X.shape[0], 1, device=X.device)
        node = 0
        offset = 0
        for level in range(self.depth):
            n_nodes = 2 ** level
            g = gate[:, offset:offset + n_nodes]                   # (n, n_nodes)
            # left child gets g, right child gets 1-g
            w = torch.stack([w * g, w * (1.0 - g)], dim=2).reshape(X.shape[0], -1)
            offset += n_nodes
        return w                                                    # (n, leaves)

    def _forward(self, X):
        return self._leaf_weights(X) @ self.leaf

    def fit(self, Z: np.ndarray, Y: np.ndarray, verbose: bool = False):
        Xt = torch.as_tensor(Z, dtype=torch.float32, device=self.device)
        Yt = torch.as_tensor(Y, dtype=torch.float32, device=self.device)
        # warm-start the leaves at the target mean so the model begins at the
        # constant-mean predictor rather than at zero, and every subsequent
        # step is a genuine improvement on it
        with torch.no_grad():
            self.leaf += Yt.mean(dim=0, keepdim=True)
        opt = torch.optim.Adam(self.params, lr=self.lr,
                               weight_decay=self.weight_decay)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.epochs)
        n = Xt.shape[0]
        for ep in range(self.epochs):
            perm = torch.randperm(n, device=self.device)
            for s in range(0, n, self.batch_size):
                idx = perm[s:s + self.batch_size]
                opt.zero_grad(set_to_none=True)
                loss = ((self._forward(Xt[idx]) - Yt[idx]) ** 2).sum(1).mean()
                loss.backward()
                opt.step()
            sched.step()
            if verbose and (ep % 50 == 0 or ep == self.epochs - 1):
                with torch.no_grad():
                    full = ((self._forward(Xt) - Yt) ** 2).sum(1).mean()
                print(f"    sdt epoch {ep:4d}  mse {full.item():.5f}  "
                      f"beta {self.beta.detach().item():.2f}")
        return self


    def predict_logits(self, Z: np.ndarray) -> np.ndarray:
        with torch.no_grad():
            Xt = torch.as_tensor(np.asarray(Z), dtype=torch.float32,
                                 device=self.device)
            out = [self._forward(Xt[s:s + 4096]).cpu().numpy()
                   for s in range(0, Xt.shape[0], 4096)]
        return np.concatenate(out, axis=0).astype(np.float64)

    def complexity(self) -> dict:
        return {"internal_nodes": self.n_internal, "leaves": self.n_leaves,
                "params": int(self.W.numel() + self.b.numel() + self.leaf.numel())}


# ---------------------------------------------------------------------------
# GA2M
# ---------------------------------------------------------------------------


class GA2MStudent:
    """One Explainable Boosting Regressor per output logit.

    Falls back to a pure-additive gradient-boosted model if `interpret` is not
    installed; the fallback is flagged in the results so it is never silently
    reported as GA2M.
    """

    def __init__(self, n_out: int, interactions: int = 20, seed: int = 0):
        self.n_out = n_out
        self.interactions = interactions
        self.seed = seed
        self.models: list = []
        self.backend = "ebm"
        try:
            from interpret.glassbox import ExplainableBoostingRegressor  # noqa
        except Exception:
            self.backend = "fallback-gbm-additive"

    def fit(self, Z: np.ndarray, Y: np.ndarray):
        Y = np.asarray(Y, dtype=np.float64)
        if Y.ndim == 1:
            Y = Y[:, None]
        self.models = []
        if self.backend == "ebm":
            from interpret.glassbox import ExplainableBoostingRegressor
            for k in range(Y.shape[1]):
                m = ExplainableBoostingRegressor(
                    interactions=self.interactions, random_state=self.seed)
                m.fit(Z, Y[:, k])
                self.models.append(m)
        else:
            from sklearn.ensemble import HistGradientBoostingRegressor
            for k in range(Y.shape[1]):
                m = HistGradientBoostingRegressor(
                    max_iter=300, max_depth=2, random_state=self.seed)
                m.fit(Z, Y[:, k])
                self.models.append(m)
        return self

    def predict_logits(self, Z: np.ndarray) -> np.ndarray:
        preds = [np.asarray(m.predict(Z), dtype=np.float64) for m in self.models]
        return np.column_stack(preds)

    def complexity(self) -> dict:
        if self.backend == "ebm" and self.models:
            m = self.models[0]
            terms = getattr(m, "term_features_", [])
            return {"shape_functions": sum(1 for t in terms if len(t) == 1),
                    "interactions": sum(1 for t in terms if len(t) == 2),
                    "components": len(terms), "backend": self.backend}
        return {"components": -1, "backend": self.backend}


### 1.8 Monte-Carlo reference Sobol' indices on the teacher

In [ ]:
"""Monte-Carlo reference Sobol' indices under the product measure mu.

These are the *teacher-side ground truth* that the students' analytic indices
are compared against.  Sampling is done by independent column resampling of
the training design, which draws exactly from mu = prod_j mu_hat_j -- the same
measure the certificate is stated under.

Total-effect indices use the Jansen estimator

    T_j = E[(f(A) - f(A_B^(j)))^2] / (2 Var f)

and first-order indices use the Saltelli/Sobol' form

    S_j = (E[f(B) (f(A_B^(j)) - f(A))]) / Var f

both summed over output dimensions so that they match the multi-output
definition used for the PCE student.
"""
from __future__ import annotations

from typing import Callable

import numpy as np


def mc_sobol_indices(f: Callable[[np.ndarray], np.ndarray], X_ref: np.ndarray,
                     n_base: int, rng: np.random.Generator,
                     batch: int = 20000) -> dict[str, np.ndarray]:
    """Jansen total effects and first-order indices for a vector-valued f.

    Costs (d + 2) * n_base evaluations of f.
    """
    d = X_ref.shape[1]
    A = product_measure_sample(X_ref, n_base, rng)
    B = product_measure_sample(X_ref, n_base, rng)

    fA = np.asarray(f(A), dtype=np.float64)
    fB = np.asarray(f(B), dtype=np.float64)
    if fA.ndim == 1:
        fA, fB = fA[:, None], fB[:, None]

    var = multioutput_variance(np.vstack([fA, fB]))
    if var <= 0:
        nan = np.full(d, np.nan)
        return {"total": nan, "first": nan.copy(), "variance": 0.0}

    total = np.empty(d)
    first = np.empty(d)
    for j in range(d):
        AB = A.copy()
        AB[:, j] = B[:, j]
        fAB = np.asarray(f(AB), dtype=np.float64)
        if fAB.ndim == 1:
            fAB = fAB[:, None]
        total[j] = ((fA - fAB) ** 2).sum(axis=1).mean() / (2.0 * var)
        first[j] = (fB * (fAB - fA)).sum(axis=1).mean() / var

    return {"total": total, "first": first, "variance": float(var)}


def mc_sobol_indices_split(f: Callable[[np.ndarray], np.ndarray],
                           X_ref: np.ndarray, n_base: int,
                           rng: np.random.Generator) -> dict:
    """Jansen indices at `n_base`, PLUS the estimator's own sampling error,
    for the price of the indices alone.

    Theorem 1 bounds the distance to the teacher's TRUE indices; we only have
    an estimate, so any measured discrepancy carries the estimator's error on
    top of the real one. The obvious way to quantify that is to run the whole
    estimator twice on independent samples -- which triples the single most
    expensive step in the pipeline, ((d+2) * n_base teacher evaluations).

    Instead we evaluate once and split the sample in half. The two half-sized
    estimates are independent, so their discrepancy measures the error at
    n_base/2; a Monte-Carlo standard error scales as n^(-1/2), so the
    difference of two independent half-estimates has standard deviation
    2x that of a single full-sample estimate. Dividing by 2 therefore gives
    the floor at the full sample size, free.
    """
    d = X_ref.shape[1]
    A = product_measure_sample(X_ref, n_base, rng)
    B = product_measure_sample(X_ref, n_base, rng)

    fA = np.asarray(f(A), dtype=np.float64)
    fB = np.asarray(f(B), dtype=np.float64)
    if fA.ndim == 1:
        fA, fB = fA[:, None], fB[:, None]

    var = multioutput_variance(np.vstack([fA, fB]))
    if var <= 0:
        nan = np.full(d, np.nan)
        return {"total": nan, "first": nan.copy(), "variance": 0.0,
                "floor_max_total": float("nan"), "floor_l1_first": float("nan")}

    h = n_base // 2
    sl = (slice(0, h), slice(h, 2 * h))
    var_h = [multioutput_variance(np.vstack([fA[s], fB[s]])) for s in sl]

    total = np.empty(d); first = np.empty(d)
    tot_h = np.empty((2, d)); fst_h = np.empty((2, d))
    for j in range(d):
        AB = A.copy()
        AB[:, j] = B[:, j]
        fAB = np.asarray(f(AB), dtype=np.float64)
        if fAB.ndim == 1:
            fAB = fAB[:, None]
        total[j] = ((fA - fAB) ** 2).sum(axis=1).mean() / (2.0 * var)
        first[j] = (fB * (fAB - fA)).sum(axis=1).mean() / var
        for k, s in enumerate(sl):
            if var_h[k] > 0:
                tot_h[k, j] = ((fA[s] - fAB[s]) ** 2).sum(axis=1).mean() / (2.0 * var_h[k])
                fst_h[k, j] = (fB[s] * (fAB[s] - fA[s])).sum(axis=1).mean() / var_h[k]
            else:
                tot_h[k, j] = fst_h[k, j] = np.nan

    dt = np.abs(tot_h[0] - tot_h[1])
    df = np.abs(np.clip(fst_h[0], 0, None) - np.clip(fst_h[1], 0, None))
    ok_t, ok_f = np.isfinite(dt), np.isfinite(df)
    return {
        "total": total, "first": first, "variance": float(var),
        # /2 rescales the half-sample discrepancy to the full-sample floor
        "floor_max_total": float(np.max(dt[ok_t]) / 2.0) if ok_t.any() else float("nan"),
        "floor_l1_first": float(np.sum(df[ok_f]) / 2.0) if ok_f.any() else float("nan"),
    }


def mc_reference_noise_floor(f: Callable[[np.ndarray], np.ndarray],
                             X_ref: np.ndarray, n_base: int,
                             rng: np.random.Generator) -> dict[str, float]:
    """Exact (2x cost) version of the floor, kept for validating the split
    estimator above. `mc_sobol_indices_split` is what the pipeline uses."""
    a = mc_sobol_indices(f, X_ref, n_base, rng)
    b = mc_sobol_indices(f, X_ref, n_base, rng)
    ta, tb = a["total"], b["total"]
    fa, fb = np.clip(a["first"], 0, None), np.clip(b["first"], 0, None)
    ok_t = np.isfinite(ta) & np.isfinite(tb)
    ok_f = np.isfinite(fa) & np.isfinite(fb)
    # two independent full-sample estimates: their difference has sd sqrt(2)x
    # that of one estimate, so divide by sqrt(2) for the single-estimate floor
    r2 = np.sqrt(2.0)
    return {
        "floor_max_total": float(np.max(np.abs(ta[ok_t] - tb[ok_t])) / r2)
                           if ok_t.any() else float("nan"),
        "floor_l1_first": float(np.sum(np.abs(fa[ok_f] - fb[ok_f])) / r2)
                          if ok_f.any() else float("nan"),
    }


def sobol_spectrum_l1(sa: dict[tuple[int, ...], float],
                      sb: dict[tuple[int, ...], float]) -> float:
    """L1 distance between two Sobol' spectra indexed by HDMR support."""
    keys = set(sa) | set(sb)
    return float(sum(abs(sa.get(k, 0.0) - sb.get(k, 0.0)) for k in keys))


def total_effects_from_spectrum(spectrum: dict[tuple[int, ...], float],
                                d: int) -> np.ndarray:
    te = np.zeros(d)
    for u, s in spectrum.items():
        for j in u:
            te[j] += s
    return te


### 1.9 Post-hoc explainers on the teacher: KernelSHAP, permutation

In [ ]:
"""Post-hoc explainers applied to the *teacher*.

The assignment asks for one post-hoc explainer on the teacher; we use
KernelSHAP (Lundberg & Lee, 2017) as the primary and permutation importance as
a cheap secondary that is affordable at d = 166 on every seed.

Both are reduced to a global feature-importance vector so that they are
comparable with total-effect Sobol' indices: mean |phi_j| over the explained
instances, summed over output dimensions.
"""
from __future__ import annotations

from typing import Callable

import numpy as np


def permutation_importance_logits(f: Callable[[np.ndarray], np.ndarray],
                                  X: np.ndarray, rng: np.random.Generator,
                                  n_repeats: int = 3) -> np.ndarray:
    """Increase in logit MSE when feature j is permuted. Model-agnostic, cheap."""
    base = np.asarray(f(X), dtype=np.float64)
    if base.ndim == 1:
        base = base[:, None]
    d = X.shape[1]
    out = np.zeros(d)
    for j in range(d):
        acc = 0.0
        for _ in range(n_repeats):
            Xp = X.copy()
            Xp[:, j] = Xp[rng.permutation(X.shape[0]), j]
            fp = np.asarray(f(Xp), dtype=np.float64)
            if fp.ndim == 1:
                fp = fp[:, None]
            acc += ((base - fp) ** 2).sum(axis=1).mean()
        out[j] = acc / n_repeats
    s = out.sum()
    return out / s if s > 0 else out


def kernel_shap_global(f: Callable[[np.ndarray], np.ndarray], X_ref: np.ndarray,
                       n_explain: int, n_background: int,
                       rng: np.random.Generator, nsamples: int | str = "auto",
                       max_d: int = 10_000) -> dict:
    """Global importance = mean |SHAP value|, aggregated over outputs.

    The background set is drawn from the product measure, not from the data,
    so that the marginalisation KernelSHAP performs matches the reference
    measure the Sobol' indices are defined under -- otherwise the two
    explanations would not be answering the same question.
    """
    if X_ref.shape[1] > max_d:
        return {"available": False,
                "reason": f"skipped: d={X_ref.shape[1]} exceeds shap_max_d="
                          f"{max_d} (cost is O(n_explain*nsamples*n_bg))",
                "importance": None}
    try:
        import shap
    except Exception as exc:
        return {"available": False, "reason": f"shap not installed ({exc})",
                "importance": None}

    background = product_measure_sample(X_ref, n_background, rng)
    idx = rng.choice(X_ref.shape[0], size=min(n_explain, X_ref.shape[0]),
                     replace=False)
    X_exp = X_ref[idx]

    try:
        explainer = shap.KernelExplainer(lambda A: np.asarray(f(A)), background)
        vals = explainer.shap_values(X_exp, silent=True, nsamples=nsamples)
    except Exception as exc:  # pragma: no cover - depends on shap version
        return {"available": False, "reason": f"kernel shap failed ({exc})",
                "importance": None}

    arr = np.asarray(vals, dtype=np.float64)
    # shap returns (n, d) for single output or (n, d, K) / list-of-(n, d)
    if arr.ndim == 3:
        imp = np.abs(arr).mean(axis=0).sum(axis=-1)
    elif arr.ndim == 2:
        imp = np.abs(arr).mean(axis=0)
    else:
        return {"available": False, "reason": f"unexpected shap shape {arr.shape}",
                "importance": None}

    s = imp.sum()
    return {"available": True, "reason": "",
            "importance": (imp / s if s > 0 else imp),
            "n_explain": int(X_exp.shape[0]),
            "n_background": int(background.shape[0])}


### 1.10 Fidelity, explanation agreement, and the certificate report

In [ ]:
"""Fidelity, explanation-agreement and certificate metrics."""
from __future__ import annotations

import numpy as np
from scipy.stats import spearmanr


# ---------------------------------------------------------------------------
# (a) fidelity
# ---------------------------------------------------------------------------


def _softmax(L: np.ndarray) -> np.ndarray:
    L = np.asarray(L, dtype=np.float64)
    L = L - L.max(axis=1, keepdims=True)
    e = np.exp(L)
    return e / e.sum(axis=1, keepdims=True)


def jensen_shannon(P: np.ndarray, Q: np.ndarray, eps: float = 1e-12) -> float:
    P = np.clip(P, eps, 1.0); Q = np.clip(Q, eps, 1.0)
    M = 0.5 * (P + Q)
    kl = lambda A, B: (A * (np.log(A) - np.log(B))).sum(axis=1)
    return float((0.5 * kl(P, M) + 0.5 * kl(Q, M)).mean())


def fidelity_metrics(teacher_logits: np.ndarray, student_logits: np.ndarray,
                     y_true: np.ndarray | None = None) -> dict:
    """All prediction-level fidelity numbers, plus the certified eta."""
    tl = np.asarray(teacher_logits, dtype=np.float64)
    sl = np.asarray(student_logits, dtype=np.float64)
    Pt, Ps = _softmax(tl), _softmax(sl)
    out = {
        "eta": relative_l2(tl, sl),
        "top1_agreement": float((tl.argmax(1) == sl.argmax(1)).mean()),
        "js_divergence": jensen_shannon(Pt, Ps),
        "logit_rmse": float(np.sqrt(((tl - sl) ** 2).sum(axis=1).mean())),
    }
    if y_true is not None:
        out["teacher_acc"] = float((tl.argmax(1) == y_true).mean())
        out["student_acc"] = float((sl.argmax(1) == y_true).mean())
        out["acc_gap"] = out["teacher_acc"] - out["student_acc"]
    return out


# ---------------------------------------------------------------------------
# (c) explanation agreement
# ---------------------------------------------------------------------------


def top_k_jaccard(a: np.ndarray, b: np.ndarray, k: int = 10) -> float:
    k = int(min(k, len(a), len(b)))
    if k == 0:
        return float("nan")
    A = set(np.argsort(-np.asarray(a))[:k])
    B = set(np.argsort(-np.asarray(b))[:k])
    return len(A & B) / len(A | B)


def rank_agreement(a: np.ndarray, b: np.ndarray, k: int = 10) -> dict:
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() < 3 or np.allclose(a[ok], a[ok][0]) or np.allclose(b[ok], b[ok][0]):
        rho = float("nan")
    else:
        rho = float(spearmanr(a[ok], b[ok]).statistic)
    return {"spearman": rho, f"jaccard_top{k}": top_k_jaccard(a, b, k)}


# ---------------------------------------------------------------------------
# (d) certificate
# ---------------------------------------------------------------------------


def certificate_report(eta_mu: float, measured_max_total: float,
                       measured_l1_first: float, mc_noise_floor: float,
                       teacher_total: np.ndarray | None = None) -> dict:
    """Bound vs. measurement, against the quantities Theorem 1 actually bounds.

    Theorem 1 bounds the distance to the teacher's TRUE Sobol' spectrum, but
    the teacher side is only ever a Monte-Carlo estimate.  So the honest test
    is `measured <= Delta(eta) + floor`, where `floor` is the reference
    estimator's own sampling error (see `mc_reference_noise_floor`).  A run
    that fails the raw test but passes the floor-corrected one is reporting a
    reference-estimation problem, not a theorem violation; a run that fails
    both is a bug.

    Two measured quantities, both genuine consequences of the theorem:
      * `measured_max_total` = max_j |T_j(f_S) - T_j(f_T)|, the second half of
        Eq. (5) -- the preferred statistic, because Jansen total effects are
        far better conditioned than first-order estimates.
      * `measured_l1_first`  = sum_j |S_j(f_S) - S_j(f_T)|, a sub-sum of the
        theorem's single l1 budget.  Both sides are normalised by their own
        TOTAL variance; renormalising the first-order part to sum to one is
        *not* the theorem's quantity and manufactures error out of nothing.

    Three mutually exclusive states, which the report must not conflate:

      `precondition_violated` (eta >= 1).  Theorem 1 assumes eta < 1 and says
        nothing whatever outside it; Delta is not a large number, it does not
        exist.  A student here is worse than the teacher's own mean under mu,
        so there is no claim to test and `bound_holds*` are None, not True.
        Reporting True because `measured <= inf` would record a trivial pass
        as a confirmation -- precisely the error Proposition 3 warns about.
      `vacuous` (1/3 <= eta < 1, so 2 <= Delta < inf).  The theorem applies
        and the bound is sound, but normalised indices lie in [0, 1] and so
        differ by at most 1 regardless; the certificate asserts nothing a
        reader did not already know.  `bound_holds` is still tested, because
        a failure here would be a genuine theorem violation (i.e. a bug).
      informative (eta < 1/3, Delta < 2).  The only state in which a pass is
        evidence of anything.  `tightness` is defined only here.
    """
    bound = certificate_bound(eta_mu)
    floor = float(mc_noise_floor) if np.isfinite(mc_noise_floor) else 0.0
    eff = bound + floor
    # eta >= 1 is a different kind of event from a large Delta: the theorem's
    # hypothesis fails, so neither the bound nor a test of it is defined.
    violated = not (np.isfinite(eta_mu) and float(eta_mu) < 1.0)
    rep = {
        "eta_mu": float(eta_mu),
        "bound_delta": bound,
        "mc_noise_floor": floor,
        "bound_plus_floor": eff,
        "measured_max_total": float(measured_max_total),
        "measured_l1_first": float(measured_l1_first),
        # raw test: ignores that the reference is itself estimated
        "bound_holds_raw": (None if violated else
                            bool(measured_max_total <= bound + 1e-9)),
        # honest test: charges the reference estimator its own error
        "bound_holds": (None if violated else
                        bool(measured_max_total <= eff + 1e-9)),
        "bound_holds_l1": (None if violated else
                           bool(measured_l1_first <= eff + 1e-9)),
        "tightness": (float(measured_max_total / bound)
                      if np.isfinite(bound) and bound > 0 else float("nan")),
        "floor_dominates": bool(np.isfinite(bound) and floor > bound),
        "precondition_violated": bool(violated),
        # vacuous means "sound but uninformative"; it does NOT cover the
        # violated case, which is tracked separately above.
        "vacuous": bool(not violated and bound >= 2.0),
    }
    if teacher_total is not None and len(teacher_total):
        vals = np.sort(np.asarray(teacher_total, dtype=np.float64))[::-1]
        pairs = total = 0
        for i in range(len(vals)):
            for j in range(i + 1, len(vals)):
                total += 1
                if vals[i] - vals[j] > eff:
                    pairs += 1
        rep["certified_pair_fraction"] = pairs / total if total else float("nan")
    return rep


def aggregate(records: list[dict], keys: list[str]) -> dict:
    """mean +/- sd over seeds for the named keys."""
    out = {}
    for k in keys:
        v = np.array([r[k] for r in records if k in r and r[k] is not None],
                     dtype=np.float64)
        v = v[np.isfinite(v)]
        out[k] = {"mean": float(v.mean()) if v.size else float("nan"),
                  "std": float(v.std(ddof=1)) if v.size > 1 else 0.0,
                  "n": int(v.size)}
    return out


### 1.11 The driver: one (dataset, teacher, rho, seed) cell end to end

In [ ]:
"""The experiment driver: one (dataset, teacher, seed, rho) cell end to end.

Order of operations in `run_cell`, which is also the order of the pipeline
figure in the report:

  1. load + quantile-transform the dataset
  2. train the teacher for cfg.epochs epochs, then freeze it
  3. build the query set Z = (1-rho) Z_data  U  rho Z_prod and label it with
     the frozen teacher's centred logits
  4. fit the three students on that same target
  5. evaluate fidelity under BOTH measures (nu = data, mu = product)
  6. compute reference explanations on the teacher (MC Sobol' + KernelSHAP)
  7. convert eta_mu into the certificate and check it against the measured
     L1 discrepancy

Results are written as one JSON file per cell so a Colab session that dies
half-way can be resumed without recomputing anything.
"""
from __future__ import annotations

import json
import os
import time
from pathlib import Path

import numpy as np


def cell_id(dataset: str, teacher: str, seed: int, rho: float) -> str:
    return f"{dataset}__{teacher}__seed{seed}__rho{rho:g}"


# Knobs added after the first release. A partial update -- a new
# experiment.py next to an older config.py, which is what happens when single
# files are copied into a Colab clone instead of the whole package -- otherwise
# dies with a bare AttributeError on the first cell that touches one, and dies
# again on every subsequent cell. The grid is hours long, so it is worth
# failing informatively once and continuing on documented defaults rather than
# burning a session on a crash loop.
_CONFIG_DEFAULTS = {
    "knee_tol": 0.05, "knee_abs_tol": 0.005, "n_bootstrap": 400,
    "teacher_min_lift": 0.02, "budget_match": False,
    "budget_match_max_d": 32, "sdt_depth_grid": (3, 4, 5, 6),
    "ga2m_interaction_grid": (0, 5, 10, 20, 40), "shap_nsamples": 256,
    "shap_max_d": 400, "perm_repeats": 3, "perm_sample": 2000,
    "sdt_epochs": 300, "sdt_beta": 4.0, "sdt_w_scale": 1.0, "sdt_lr": 2e-2,
    "student_min_fit": 0.95, "fit_holdout_frac": 0.2, "max_cond": 1e10,
}
_CONFIG_WARNED = False


def _resolve_config(cfg):
    """Fill in any knob this module needs that `cfg` predates, once, loudly.

    Returns cfg with the gaps filled; the names filled in are recorded on
    `cfg._defaulted` so each run record can state them. An older config.py is
    not a dataclass holding these fields, so `as_dict()` would omit them
    entirely and Table 5 would report the grid as though every value had been
    chosen deliberately.
    """
    global _CONFIG_WARNED
    stale = [k for k in _CONFIG_DEFAULTS if not hasattr(cfg, k)]
    try:
        cfg._defaulted = list(stale)
    except Exception:
        pass
    if not stale:
        return cfg
    for k in stale:
        setattr(cfg, k, _CONFIG_DEFAULTS[k])
    if not _CONFIG_WARNED:
        _CONFIG_WARNED = True
        print("=" * 72)
        print("[CONFIG] pact/config.py is older than pact/experiment.py.")
        print(f"[CONFIG] Missing {len(stale)} setting(s), now defaulted:")
        for k in stale:
            print(f"[CONFIG]    {k} = {_CONFIG_DEFAULTS[k]!r}")
        print("[CONFIG] The run will proceed, but update the WHOLE package")
        print("[CONFIG] (git pull, or re-upload the repo) rather than copying")
        print("[CONFIG] single files -- Table 5 reads the config off these")
        print("[CONFIG] records, so a defaulted knob is reported as if chosen.")
        print("=" * 72)
    return cfg


def _split_rows(X_train, holdout_frac: float, seed: int):
    """Disjoint (fitting pool, selection pool) over the training ROWS.

    `holdout_frac = 0` returns the full set twice, i.e. the earlier behaviour
    in which the operating point was chosen on rows the student had seen.
    Anything above 0 makes the held-out claim true. The permutation depends
    only on `seed`, so the split is part of the reproducible cell state.
    """
    n = X_train.shape[0]
    if not holdout_frac or holdout_frac <= 0.0:
        return X_train, X_train
    n_sel = int(round(min(max(holdout_frac, 0.0), 0.5) * n))
    if n_sel < 2:
        return X_train, X_train
    perm = np.random.default_rng(10_000 + int(seed)).permutation(n)
    return X_train[perm[n_sel:]], X_train[perm[:n_sel]]


def _build_query_set(X_train, rho, n_query, rng):
    """A query set of exactly `n_query` rows, a fraction `rho` of them from mu.

    The data part is drawn with replacement when more rows are requested than
    the pool holds. An earlier version capped it at `size=min(n_data, n_rows)`
    while setting `replace` correctly, which silently returned FEWER rows than
    asked for and, at rho>0, a mixture that was not rho: with n_query=12000
    against gas-drift's ~9.4k training rows, the rho=0 query set was 9389 rows
    rather than 12000, and the recorded config said 12000. The cap is gone;
    `rho` is now honoured exactly at every pool size, which is what the
    ablation table's rho column has to mean for the table to be readable.
    """
    n_prod = int(round(rho * n_query))
    n_data = n_query - n_prod
    n_rows = X_train.shape[0]
    parts = []
    if n_data > 0:
        if n_rows == 0:
            raise ValueError("empty row pool for the data component of Z")
        idx = rng.choice(n_rows, size=n_data, replace=n_data > n_rows)
        parts.append(X_train[idx])
    if n_prod > 0:
        parts.append(product_measure_sample(X_train, n_prod, rng))
    Z = np.vstack(parts)
    assert Z.shape[0] == n_query, (Z.shape[0], n_query)
    return Z


def _pick_operating_point(path, eta_by_fit=None, tol=0.05, abs_tol=0.005,
                          target_components=None):
    """Knee of the lambda path: the SMALLEST NON-EMPTY model within tolerance.

    `eta_by_fit[i]` is the held-out residual of `path[i]` on the measure the
    student was FITTED on (the rho-mixture) -- NOT under mu. That distinction
    is the whole point and an earlier docstring here described the withdrawn
    behaviour, so to be explicit:

      * selecting on eta_mu is circular (it tunes the model on the measure the
        certificate is then claimed under) and, at rho=0, degenerate: the
        empty model is eta_mu-OPTIMAL: for a constant predictor c,
        eta^2 = 1 + ||E f_T - c||^2 / sigma^2, so the intercept scores
        eta_mu >= 1 (strictly above it whenever nu and mu give f_T different
        means, which they do for a non-additive teacher), and every
        non-trivial model extrapolates worse still. argmin then returns the
        EMPTY model and the cell reports eta_fit = 1.000;
      * held-out data from the fitting distribution is the only non-circular
        basis for choosing lambda, and the certificate is evaluated under mu
        afterwards at that fixed choice.

    `active_components > 0` is a hard filter, not a tie-break: the empty model
    is the constant predictor and has no HDMR to explain, so it is never a
    valid operating point regardless of its residual. The identity of this
    rule is recorded in every run record as `provenance.SELECTION_RULE`.

    Taking the least-regularised end of the path instead reports the *largest*
    model, which overstates the component count and understates what sparsity
    costs.
    """
    if target_components is not None:
        return min(path, key=lambda f: abs(f.active_components - target_components))
    if not eta_by_fit:
        return path[-1]
    e = np.asarray(eta_by_fit, dtype=np.float64)
    ok = np.isfinite(e)
    if not ok.any():
        return path[-1]
    best = float(np.nanmin(e))
    # Relative tolerance alone collapses as the best residual approaches zero:
    # 5% of 0.001 admits nothing but the single best point, so the "smallest
    # model within tolerance" rule degenerates into "largest model on the
    # path". The absolute floor keeps the rule meaningful at both ends.
    thresh = best * (1.0 + tol) + abs_tol
    cand = [i for i in range(len(path))
            if ok[i] and e[i] <= thresh and path[i].active_components > 0]
    if not cand:
        # no non-empty model qualifies: fall back to the best-fitting one that
        # actually has components, and only then to the raw argmin
        nonempty = [i for i in range(len(path))
                    if ok[i] and path[i].active_components > 0]
        if nonempty:
            return path[min(nonempty, key=lambda i: e[i])]
        return path[int(np.nanargmin(e))]
    return path[min(cand, key=lambda i: path[i].active_components)]


def _selection_verdict(path, eta_fit_path, best, thresh: float = 0.95) -> dict:
    """Is a degenerate operating point a training failure or a selection one?

    These are opposite diagnoses and the earlier code could not tell them
    apart. When the reported student is the intercept-only model its residual
    on the fitting measure is eta_fit = 1.000 (exactly the constant-mean
    predictor), and the old message read "did not beat the constant-mean
    predictor ... row invalid" -- blaming the optimiser. But if some OTHER
    point on the same lambda path has eta_fit well below 1, the optimiser
    worked fine and the *selection rule* returned the wrong point. That is a
    bug in the reporting layer, not a property of the data, and it must not be
    recorded as evidence of anything.

    This is precisely what produced the irreproducible pair

        eta_fit = 0.136   (selection on the fitting measure)
        eta_fit = 1.000   (selection on mu -- intercept-only model returned)

    for one deterministic fit. Returns a dict so the run record carries the
    distinction instead of a single ambiguous boolean.
    """
    e = np.asarray(eta_fit_path, dtype=np.float64)
    finite = np.isfinite(e)
    best_available = float(np.nanmin(e)) if finite.any() else float("nan")
    # the best NON-EMPTY model the path actually contains
    nonempty = [i for i in range(len(path))
                if finite[i] and path[i].active_components > 0]
    best_nonempty = float(min(e[i] for i in nonempty)) if nonempty else float("nan")
    degenerate = int(best.active_components) == 0
    path_has_valid = bool(np.isfinite(best_nonempty) and best_nonempty < thresh)

    # Solver and grid state, so a failing cell carries its own diagnosis
    # instead of needing the fit re-run to find out what happened.
    lams = [float(f.lam) for f in path]
    n_unconverged = sum(1 for f in path if not bool(getattr(f, "converged", True)))
    max_components = max((int(f.active_components) for f in path), default=0)

    if degenerate and path_has_valid:
        kind = "selection_failure"
        note = (f"operating point is the intercept-only model (eta_fit=1.000 "
                f"by construction) while the path contains a non-empty model "
                f"with eta_fit={best_nonempty:.3f}. The optimiser did not "
                f"fail; the selection rule returned the wrong point. This row "
                f"is a reporting bug, not a result -- do not interpret it.")
    elif not np.isfinite(best_available):
        # Checked BEFORE the empty-path branch: `nonempty` requires a finite
        # residual, so an all-NaN path over a model that does have components
        # would otherwise be misreported as "no component ever enters".
        kind = "numerical_failure"
        note = (f"the lambda path contains no finite residual over "
                f"{len(path)} points (up to {max_components} components, "
                f"{n_unconverged} at the iteration cap): the design or the "
                f"solve overflowed, so no operating point is interpretable")
    elif not nonempty:
        # No point on the grid ever admits a component. The student never
        # entered the model class, so this says nothing about the teacher.
        kind = "empty_path"
        note = (f"every one of the {len(path)} lambda points is the "
                f"intercept-only model: no component ever enters over "
                f"lambda in [{min(lams):.3g}, {max(lams):.3g}]. That is a "
                f"grid/solver failure, not a property of the teacher -- "
                f"lower lambda_min_ratio or raise max_iter and refit "
                f"({n_unconverged}/{len(path)} lambdas hit the iteration cap).")
    elif not path_has_valid:
        # THIS is the state the earlier code mislabelled "ok" while the
        # training gate simultaneously called the row invalid: the operating
        # point is non-empty, so `degenerate` is False, yet nothing on the
        # whole path beats the constant-mean predictor. Reaching the `else`
        # branch here reported success for a cell that had none.
        kind = "no_signal"
        note = (f"no point on the path beats the fitting criterion "
                f"(best non-empty eta_fit={best_nonempty:.3f} >= {thresh:g} "
                f"over lambda in [{min(lams):.3g}, {max(lams):.3g}], up to "
                f"{max_components} components, "
                f"{n_unconverged}/{len(path)} at the iteration cap). Either "
                f"the teacher carries no order-<={path[0].max_order or 2} "
                f"signal on this measure, or lambda_min_ratio is too large / "
                f"max_iter too small -- the convergence count separates those.")
    else:
        kind = "ok"
        note = (f"non-empty operating point, best non-empty eta_fit on path "
                f"={best_nonempty:.3f}")
    return {"kind": kind, "note": note, "degenerate": degenerate,
            "best_eta_fit_on_path": best_available,
            "best_nonempty_eta_fit_on_path": best_nonempty,
            "path_has_valid_model": path_has_valid,
            "n_nonempty_points": len(nonempty),
            "n_path_points": len(path),
            "max_active_components": max_components,
            "n_unconverged": n_unconverged,
            "lambda_min": min(lams) if lams else float("nan"),
            "lambda_max": max(lams) if lams else float("nan")}


def _training_verdict(name: str, fid_fit: dict, fid_mu: dict,
                      rho: float, thresh: float = 0.95,
                      selection: dict | None = None) -> tuple[bool, str]:
    """Separate "never trained" from "trained, then diverged off-manifold".

    eta = 1.0 is exactly the constant-mean predictor, so eta >= 1 means a
    student carries no information about the teacher *on the measure eta was
    computed under*. Judging that on mu alone conflates two opposite findings:

      * at rho = 1 the student was fitted on mu, so eta_mu >= 1 is a genuine
        optimisation failure and its row must not enter a comparison;
      * at rho = 0 the student was fitted on the data manifold and is
        unconstrained off it, so eta_mu >= 1 while the fitted-measure residual
        is small is exactly Proposition 3 -- the headline prediction of this
        experiment, and a result to report rather than a row to discard.

    The verdict therefore reads the residual on a held-out sample from the same
    rho-mixture the student was fitted on.
    """
    e_fit, e_mu = fid_fit.get("eta", float("nan")), fid_mu.get("eta", float("nan"))
    trained = bool(np.isfinite(e_fit) and e_fit < thresh)
    if not trained:
        # A failed fitting criterion has two possible causes and they must not
        # be reported with the same words. If the lambda path contains a good
        # non-empty model, the optimiser worked and the selection rule is at
        # fault -- saying "did not beat the constant-mean predictor" there
        # blames the data for a bug in the reporting layer.
        if selection and selection.get("kind") == "selection_failure":
            note = f"SELECTION FAILURE: {selection['note']}"
            print(f"  [BUG]  {name} {note}")
            return False, note
        # The bare message below asserts a cause it has not established: it
        # reads as "the student cannot fit the teacher" when the actual cause
        # may be an all-empty lambda path, an iteration cap, or a rank-deficient
        # design. The selection verdict knows which, and discarding it here was
        # what made the gas-drift/rho=0 warning unactionable -- it named a
        # conclusion and withheld every number behind it.
        note = (f"did not beat the constant-mean predictor on its own fitting "
                f"measure (eta_fit={e_fit:.3f} at rho={rho:g}); row invalid")
        if selection and selection.get("note"):
            note = f"{note} -- cause [{selection.get('kind')}]: {selection['note']}"
        print(f"  [WARN] {name} {note}")
    elif np.isfinite(e_mu) and e_mu >= 1.0:
        note = (f"trained (eta_fit={e_fit:.3f}) but carries no information "
                f"under mu (eta_mu={e_mu:.3f}): off-manifold divergence, "
                f"Prop. 3 -- expected at rho={rho:g}, not a failure")
        print(f"  [note] {name} {note}")
    else:
        note = f"ok (eta_fit={e_fit:.3f}, eta_mu={e_mu:.3f})"
    return trained, note


def _bootstrap_eta(Y_true: np.ndarray, Y_pred: np.ndarray, rng,
                   n_boot: int = 400, alpha: float = 0.05) -> dict:
    """Bootstrap CI for eta-hat over the held-out sample, and the induced
    interval on Delta(eta).

    The certificate is a deterministic function of a quantity we estimate from
    a finite sample, so a point estimate of Delta overstates what was shown.
    Resampling the held-out sample (not the seeds) isolates that one source of
    error. The paper's Limitations section claims these intervals; this is
    where they come from.
    """
    n = Y_true.shape[0]
    if n < 32:
        return {}
    etas = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        etas[b] = relative_l2(Y_true[idx], Y_pred[idx])
    etas = etas[np.isfinite(etas)]
    if etas.size < 10:
        return {}
    lo, hi = np.percentile(etas, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return {"eta_lo": float(lo), "eta_hi": float(hi),
            "delta_lo": certificate_bound(float(lo)),
            "delta_hi": certificate_bound(float(hi)),
            "n_boot": int(etas.size)}


def _write_record_atomic(path: Path, rec: dict) -> None:
    """Write a cell record so a killed session cannot leave a half-file.

    A Colab runtime that dies mid-`write_text` leaves a truncated JSON on
    disk. That file still satisfies `path.exists()`, so the cell is skipped
    forever on resume, and `json.loads` on it takes down the whole analysis
    notebook. Writing to a sibling temp file and renaming makes the record
    appear whole or not at all (`os.replace` is atomic within a filesystem).
    """
    tmp = path.with_name(path.name + ".partial")
    try:
        tmp.write_text(json.dumps(rec, indent=1))
        os.replace(tmp, path)
    finally:
        if tmp.exists():
            try:
                tmp.unlink()
            except OSError:
                pass


def read_record(path: Path) -> dict | None:
    """Parse a cell record, or None if it is absent, corrupt or truncated.

    Used by both the resume check and `load_runs`, so a damaged file is
    treated identically in both: re-run it, and leave it out of the analysis.
    """
    try:
        rec = json.loads(Path(path).read_text())
    except (OSError, ValueError):
        return None
    if not isinstance(rec, dict) or "students" not in rec or "cell" not in rec:
        return None
    return rec


# The teacher depends on (dataset, kind, seed, architecture) and NOT on rho:
# rho selects the student's query set, long after the teacher is frozen. The
# grid sweeps rho in the inner loop, so without this cache the same teacher is
# retrained once per rho -- four identical trainings per (dataset, seed) over
# RHO_ABLATION, which is the second largest avoidable cost in the run after
# refetching the data.
#
# Caching also makes the ablation's central claim literally true rather than
# merely probable: every rho in a (dataset, seed) cell is now compared against
# the byte-identical teacher, so a difference in eta cannot be an artefact of
# teacher retraining. set_seed(seed) already made this deterministic; the
# cache makes it structural.
#
# Keyed on the config fields the teacher actually reads, so a config change
# mid-session cannot silently serve a teacher trained under the old settings.
_TEACHER_CACHE: dict[tuple, object] = {}


def _teacher_key(dataset_key, teacher_kind, seed, cfg):
    return (dataset_key, teacher_kind, int(seed), tuple(cfg.hidden),
            float(cfg.dropout), float(cfg.lr), float(cfg.weight_decay),
            int(cfg.epochs), int(cfg.batch_size), float(cfg.test_size),
            float(cfg.val_size))


def _get_teacher(dataset_key, teacher_kind, seed, cfg, ds):
    key = _teacher_key(dataset_key, teacher_kind, seed, cfg)
    hit = _TEACHER_CACHE.get(key)
    if hit is not None:
        return hit
    teacher = build_teacher(teacher_kind, ds.d, ds.n_classes, cfg, seed)
    teacher.fit(ds.X_train, ds.y_train, ds.X_val, ds.y_val)
    # one teacher per (dataset, seed) is a few MB; a 4-dataset x 10-seed grid
    # would hold 40, so keep only the most recent few
    if len(_TEACHER_CACHE) >= 4:
        _TEACHER_CACHE.pop(next(iter(_TEACHER_CACHE)))
    _TEACHER_CACHE[key] = teacher
    return teacher


def clear_teacher_cache() -> None:
    """Drop cached teachers (free GPU memory between datasets)."""
    _TEACHER_CACHE.clear()


def run_cell(dataset_key: str, seed: int, cfg: RunConfig, rho: float | None = None,
             teacher_kind: str | None = None, out_dir: str | None = None,
             overwrite: bool = False, verbose: bool = True) -> dict:
    cfg = _resolve_config(cfg)
    rho = cfg.rho if rho is None else rho
    teacher_kind = cfg.teacher if teacher_kind is None else teacher_kind
    out_dir = Path(out_dir or cfg.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    path_json = out_dir / f"{cell_id(dataset_key, teacher_kind, seed, rho)}.json"
    if path_json.exists() and not overwrite:
        cached = read_record(path_json)
        if cached is not None:
            if verbose:
                print(f"  [skip] {path_json.name} exists")
            return cached
        # present but unreadable: a session died mid-write. Re-run it rather
        # than skipping it forever on the strength of the filename.
        if verbose:
            print(f"  [redo] {path_json.name} is corrupt or truncated")

    t0 = time.time()
    # per-stage timing: the wall-clock table in the report needs it, and
    # extrapolating a 10-seed grid from a pilot cell is guesswork without it
    stage: dict[str, float] = {}
    _mark = [time.time()]
    def tick(name):
        now = time.time()
        stage[name] = stage.get(name, 0.0) + (now - _mark[0])
        _mark[0] = now
    set_seed(seed)
    rng = np.random.default_rng(seed)
    spec = DATASETS_BY_KEY[dataset_key]
    ds = load_dataset(spec, seed=seed, test_size=cfg.test_size,
                      val_size=cfg.val_size)

    # -- 2. teacher --------------------------------------------------------
    tick("load_data")
    teacher = _get_teacher(dataset_key, teacher_kind, seed, cfg, ds)
    f_teacher = teacher.predict_logits
    tick("train_teacher")

    # -- 3. query set ------------------------------------------------------
    # Partition the training ROWS before drawing anything, so the sample the
    # operating point is selected on is genuinely disjoint from the sample the
    # student is fitted on. Drawing both from the whole pool makes selection
    # in-sample whenever n_query approaches |X_train| -- which it does on the
    # two high-dimensional sets -- and the knee rule's claim to be a
    # non-circular basis for lambda then fails. The permutation is seeded, so
    # the split is reproducible and recorded.
    X_pool, X_sel = _split_rows(ds.X_train, cfg.fit_holdout_frac, seed)
    Z = _build_query_set(X_pool, rho, cfg.n_query, rng)
    Y = f_teacher(Z)

    # evaluation samples: one on the data measure nu, one on the product mu
    Z_nu = ds.X_test
    Y_nu = f_teacher(Z_nu)
    Z_mu = product_measure_sample(ds.X_train, cfg.n_holdout_mu, rng)
    Y_mu = f_teacher(Z_mu)
    # A third held-out sample from the SAME rho-mixture the student is fitted
    # on. Without it there is no way to tell "this student never trained" from
    # "this student trained and then diverged off-manifold" -- and at rho=0
    # those look identical under mu while meaning opposite things. The first is
    # a bug; the second is Prop. 3, which this experiment exists to measure.
    Z_fit = _build_query_set(X_sel, rho, cfg.n_holdout_mu, rng)
    Y_fit = f_teacher(Z_fit)
    tick("build_queries")

    rec: dict = {
        "cell": cell_id(dataset_key, teacher_kind, seed, rho),
        "dataset": dataset_key, "seed": seed, "rho": rho,
        "teacher_kind": teacher_kind,
        "dataset_summary": ds.summary(),
        "config": cfg.as_dict(),
        # any knob this module had to default because config.py was older;
        # empty on a consistent install, and Table 5 must not present a
        # defaulted value as a chosen one
        "config_defaulted": list(getattr(cfg, "_defaulted", []) or []),
        # which code and which configuration produced these numbers. Two
        # records that disagree can now be told apart before anyone asks why;
        # see pact/provenance.describe_mismatch.
        "provenance": run_provenance(cfg),
        # realised sample sizes, so a silently truncated query set is visible
        # in the record rather than inferable only from the code version
        "query_set_size": int(Z.shape[0]),
        "fit_holdout_size": int(Z_fit.shape[0]),
        "row_split": {"fit_pool": int(X_pool.shape[0]),
                      "selection_pool": int(X_sel.shape[0]),
                      "disjoint": bool(cfg.fit_holdout_frac > 0)},
        "torch": HAS_TORCH,
        "students": {},
    }

    # -- 6. teacher-side reference explanations ----------------------------
    # One pass gives both the reference indices and the estimator's own error
    # (see mc_sobol_indices_split); estimating the floor separately would
    # triple the most expensive step in the cell.
    mc = mc_sobol_indices_split(f_teacher, ds.X_train, cfg.sobol_mc_base, rng)
    tick("mc_sobol")
    teacher_total = mc["total"]
    teacher_acc = float((f_teacher(ds.X_test).argmax(1) == ds.y_test).mean())
    # A seed whose teacher failed to train poisons every downstream number in
    # the cell, so gate it here rather than discovering it in a table.
    majority = float(np.bincount(ds.y_test).max() / ds.y_test.size)
    teacher_ok = bool(teacher_acc >= majority + cfg.teacher_min_lift)
    if not teacher_ok:
        print(f"  [WARN] teacher accuracy {teacher_acc:.3f} is within "
              f"{cfg.teacher_min_lift} of the majority rate {majority:.3f}; "
              f"this seed should be excluded from the analysis")
    rec["teacher"] = {
        "acc_test": teacher_acc,
        "majority_rate": majority,
        "trained": teacher_ok,
        "mc_total_sobol": teacher_total.tolist(),
        "mc_first_sobol": mc["first"].tolist(),
        "mc_variance": mc["variance"],
        "mc_noise_floor": {"floor_max_total": mc["floor_max_total"],
                           "floor_l1_first": mc["floor_l1_first"]},
    }
    perm = permutation_importance_logits(f_teacher, Z_mu[:cfg.perm_sample],
                                         rng, n_repeats=cfg.perm_repeats)
    tick("perm_teacher")
    rec["teacher"]["permutation_importance"] = perm.tolist()
    shap_res = kernel_shap_global(f_teacher, ds.X_train, cfg.shap_n_explain,
                                  cfg.shap_n_background, rng,
                                  nsamples=cfg.shap_nsamples,
                                  max_d=cfg.shap_max_d)
    rec["teacher"]["kernelshap_available"] = shap_res["available"]
    rec["teacher"]["kernelshap_note"] = shap_res["reason"]
    rec["teacher"]["kernelshap_importance"] = (
        shap_res["importance"].tolist() if shap_res["available"] else None)
    tick("kernelshap")

    # -- 4/5/7. GAS-PCE ----------------------------------------------------
    pce = GasPCE(d=ds.d, degree=cfg.degree, q_norm=cfg.q_norm,
                 screen_m=cfg.screen_m, gamma=cfg.gamma,
                 max_order=cfg.max_order, n_lambda=cfg.n_lambda,
                 lambda_min_ratio=cfg.lambda_min_ratio,
                 max_iter=cfg.max_iter, tol=cfg.tol,
                 max_cond=cfg.max_cond).fit(Z, Y)
    tick("fit_pce")

    # Build each evaluation design matrix ONCE. predict_logits() rebuilds it
    # per call, and the lambda path calls it 20+ times on the same sample --
    # at d=166 that matrix is the single largest array in the pipeline, so
    # rebuilding it dominated the wall-clock of a high-dimensional cell.
    Psi_mu = pce.design_for(Z_mu)
    Psi_nu = pce.design_for(Z_nu)
    Psi_fit = pce.design_for(Z_fit)

    frontier, eta_path, eta_fit_path = [], [], []
    for fit in pce.path:
        fm = fidelity_metrics(Y_mu, pce.predict_from_design(Psi_mu, fit))
        ff = fidelity_metrics(Y_fit, pce.predict_from_design(Psi_fit, fit))
        eta_path.append(fm["eta"])
        eta_fit_path.append(ff["eta"])
        frontier.append({"lambda": fit.lam,
                         "active_components": fit.active_components,
                         "nnz": fit.nnz, "max_order": fit.max_order,
                         "eta_mu": fm["eta"], "eta_fit": ff["eta"],
                         "top1_agreement": fm["top1_agreement"]})
    rec["pce_frontier"] = frontier
    rec["pce_screened_features"] = pce.screened_features_.tolist()

    # Select on the measure the student was FITTED on, certify on mu.
    # Selecting on eta_mu is circular and, at rho=0, degenerate: off the data
    # manifold every non-trivial model extrapolates badly, while the
    # intercept-only model is eta_mu-optimal (it scores 1 + the squared
    # nu-vs-mu mean offset, and non-trivial models do worse). The knee
    # rule then returns the empty model -- zero active components, no
    # explanation at all -- and reports it as the operating point. Held-out
    # data from the fitting distribution is the only non-circular basis for
    # choosing lambda; the certificate is then evaluated under mu at that
    # choice, which is the honest order of operations.
    best = _pick_operating_point(pce.path, eta_fit_path, tol=cfg.knee_tol,
                                 abs_tol=cfg.knee_abs_tol)
    best_idx = pce.path.index(best)
    selection = _selection_verdict(pce.path, eta_fit_path, best,
                                   thresh=cfg.student_min_fit)
    rec["pce_operating_point"] = {
        "lambda": best.lam,
        "rule": SELECTION_RULE,
        "rule_description": f"smallest non-empty model with held-out eta on "
                f"the FITTING measure (rho={rho:g}) within "
                f"{cfg.knee_tol:.0%} + {cfg.knee_abs_tol:g} of the path best",
        "index": best_idx,
        "active_components": int(best.active_components),
        # solver state at the reported point, so a suspect number can be
        # traced to its own solve rather than to the edit history
        "solver": {"status": best.status, "n_iter": best.n_iter,
                   "converged": bool(best.converged),
                   "objective": best.objective,
                   "train_mse": best.train_mse},
        "selection": selection,
    }
    rec["pce_conditioning"] = pce.conditioning_
    # An all-zero model is the constant predictor: it has no explanation to
    # certify, so say so rather than reporting indices for an empty HDMR.
    if selection["degenerate"]:
        rec["pce_operating_point"]["degenerate"] = True
        if selection["kind"] == "selection_failure":
            print(f"  [BUG]  GAS-PCE {selection['note']}")
        else:
            print(f"  [WARN] GAS-PCE operating point has 0 active components "
                  f"(every group zeroed); lambda_min_ratio may be too large, "
                  f"or the teacher may carry no order-<={cfg.max_order} "
                  f"signal here")
    cond = pce.conditioning_ or {}
    rec["pce_pruned_terms"] = pce.pruned_terms_.tolist()
    if cond.get("pruned"):
        # Not a warning: the data did not identify those directions, so the
        # coefficients there were arbitrary and the Sobol' indices read off
        # them were too. Dropping them is what makes the cell reproducible
        # across BLAS builds; see tools/test_conditioning_uniqueness.py.
        print(f"  [note] GAS-PCE dropped {cond['pruned']} unidentified basis "
              f"term(s) (rank {cond['rank']} of {cond['n_terms']}, cond="
              f"{cond['cond']:.3g}) -> {cond['post_prune_n_terms']} terms, "
              f"cond={cond['post_prune_cond']:.3g}, "
              f"deficient={cond['post_prune_rank_deficient']}")
        if cond.get("post_prune_rank_deficient"):
            print(f"  [WARN] GAS-PCE design is STILL rank deficient after "
                  f"pruning; coefficients in the remaining null space are not "
                  f"identified and may differ between BLAS implementations")
    elif cond.get("rank_deficient"):
        print(f"  [WARN] GAS-PCE design is rank deficient "
              f"(rank {cond['rank']} of {cond['n_terms']}, cond="
              f"{cond['cond']:.3g}) and pruning is disabled "
              f"(cfg.max_cond); coefficients in the null space are not "
              f"identified and may differ between BLAS implementations")
    pce_mu = pce.predict_from_design(Psi_mu, best)
    pce_nu = pce.predict_from_design(Psi_nu, best)
    pce_fit_pred = pce.predict_from_design(Psi_fit, best)
    pce_te = best.total_effects
    tick("eval_pce")

    # --- certificate measurement -----------------------------------------
    # Theorem 1 bounds quantities normalised by TOTAL variance. An earlier
    # version of this code renormalised the first-order sub-spectrum on each
    # side to sum to one, which is NOT the theorem's quantity: when teacher
    # and student hold different shares of their variance in interactions,
    # that rescaling manufactures a discrepancy out of nothing. Both
    # statistics below keep the total-variance normalisation.
    #
    #   measured_max_total = max_j |T_j(f_S) - T_j(f_T)|   (Eq. 5, 2nd half)
    #   measured_l1_first  = sum_j |S_j(f_S) - S_j(f_T)|   (sub-sum of the
    #                                                       single l1 budget)
    measured_max_total = float(
        np.max(np.abs(pce_te - teacher_total))) if ds.d else float("nan")
    s1 = {u: s for u, s in best.sobol.items() if len(u) == 1}
    measured_l1_first = float(sum(
        abs(s1.get((j,), 0.0) - max(float(mc["first"][j]), 0.0))
        for j in range(ds.d)))

    # the reference indices are themselves estimates; charge them their error
    floor = {"floor_max_total": mc["floor_max_total"],
             "floor_l1_first": mc["floor_l1_first"]}

    pce_fid_mu = fidelity_metrics(Y_mu, pce_mu)
    pce_fid_fit = fidelity_metrics(Y_fit, pce_fit_pred)
    pce_trained, pce_note = _training_verdict("GAS-PCE", pce_fid_fit,
                                              pce_fid_mu, rho,
                                              thresh=cfg.student_min_fit,
                                              selection=selection)
    rec["students"]["gas_pce"] = {
        "fidelity_mu": pce_fid_mu,
        "fidelity_nu": fidelity_metrics(Y_nu, pce_nu, ds.y_test),
        "fidelity_fit": pce_fid_fit,
        "trained": pce_trained,
        "diagnosis": pce_note,
        "off_manifold_blowup": bool(pce_fid_mu["eta"] >= 1.0
                                    and pce_fid_fit["eta"] < 0.95),
        "eta_bootstrap": _bootstrap_eta(Y_mu, pce_mu, rng, cfg.n_bootstrap),
        "complexity": {"active_components": best.active_components,
                       "nnz": best.nnz, "max_order": best.max_order,
                       "basis_size": len(pce.index_set)},
        "total_effects": pce_te.tolist(),
        "converged": bool(getattr(pce, "all_converged_", True)),
        "agreement_vs_mc_sobol": rank_agreement(pce_te, teacher_total),
        "agreement_vs_permutation": rank_agreement(pce_te, perm),
        "agreement_vs_shap": (rank_agreement(pce_te, shap_res["importance"])
                              if shap_res["available"] else None),
        "certificate": certificate_report(
            pce_fid_mu["eta"], measured_max_total, measured_l1_first,
            floor["floor_max_total"], teacher_total),
        "mc_noise_floor": floor,
    }

    # -- SDT ---------------------------------------------------------------
    # Budget matching: comparing a depth-5 tree against whatever component
    # count the lasso happened to select is not a controlled comparison. When
    # enabled we sweep the baseline's own complexity knob and keep the setting
    # whose component count lands closest to the GAS-PCE operating point.
    target_components = best.active_components
    do_match = bool(cfg.budget_match and ds.d <= cfg.budget_match_max_d)
    rec["budget_matched"] = do_match
    if HAS_TORCH:
        depths = list(cfg.sdt_depth_grid) if do_match else [cfg.sdt_depth]
        sdt, sdt_depth_used = None, cfg.sdt_depth
        best_gap = float("inf")
        for dep in depths:
            cand = SoftDecisionTree(d=ds.d, n_out=Y.shape[1], depth=dep,
                                    seed=seed, beta=cfg.sdt_beta,
                                    w_scale=cfg.sdt_w_scale, lr=cfg.sdt_lr,
                                    epochs=cfg.sdt_epochs).fit(Z, Y)
            gap = abs(cand.complexity()["internal_nodes"] - target_components)
            if gap < best_gap:
                sdt, sdt_depth_used, best_gap = cand, dep, gap
        sdt_te = permutation_importance_logits(
        sdt.predict_logits, Z_mu[:cfg.perm_sample], rng, n_repeats=cfg.perm_repeats)
        sdt_fid = fidelity_metrics(Y_mu, sdt.predict_logits(Z_mu))
        sdt_fit = fidelity_metrics(Y_fit, sdt.predict_logits(Z_fit))
        sdt_trained, sdt_note = _training_verdict("SDT", sdt_fit, sdt_fid, rho,
                                              thresh=cfg.student_min_fit)
        rec["students"]["sdt"] = {
            "fidelity_mu": sdt_fid,
            "fidelity_nu": fidelity_metrics(Y_nu, sdt.predict_logits(Z_nu),
                                            ds.y_test),
            "fidelity_fit": sdt_fit,
            "trained": sdt_trained,
            "diagnosis": sdt_note,
            "off_manifold_blowup": bool(sdt_fid["eta"] >= 1.0
                                        and sdt_fit["eta"] < 0.95),
            "complexity": {**sdt.complexity(), "depth": sdt_depth_used,
                           "budget_matched": do_match,
                           "target_components": int(target_components)},
            "eta_bootstrap": _bootstrap_eta(Y_mu, sdt.predict_logits(Z_mu),
                                            rng, cfg.n_bootstrap),
            "total_effects": sdt_te.tolist(),
            "agreement_vs_mc_sobol": rank_agreement(sdt_te, teacher_total),
            "agreement_vs_permutation": rank_agreement(sdt_te, perm),
            "agreement_vs_shap": (rank_agreement(sdt_te, shap_res["importance"])
                                  if shap_res["available"] else None),
        }
    else:
        rec["students"]["sdt"] = {"skipped": "torch unavailable"}
    tick("sdt")

    # -- GA2M --------------------------------------------------------------
    grid = list(cfg.ga2m_interaction_grid) if do_match else [cfg.ga2m_interactions]
    ga2m, ga2m_inter, best_gap = None, cfg.ga2m_interactions, float("inf")
    for k_int in grid:
        cand = GA2MStudent(n_out=Y.shape[1], interactions=k_int,
                           seed=seed).fit(Z, Y)
        comp = cand.complexity().get("components", -1)
        gap = abs(comp - target_components) if comp > 0 else float("inf")
        if gap < best_gap or ga2m is None:
            ga2m, ga2m_inter, best_gap = cand, k_int, gap
    ga2m_mu = ga2m.predict_logits(Z_mu)
    ga2m_te = permutation_importance_logits(
        ga2m.predict_logits, Z_mu[:cfg.perm_sample], rng, n_repeats=cfg.perm_repeats)
    ga2m_fid_mu = fidelity_metrics(Y_mu, ga2m_mu)
    ga2m_fid_fit = fidelity_metrics(Y_fit, ga2m.predict_logits(Z_fit))
    ga2m_trained, ga2m_note = _training_verdict("GA2M", ga2m_fid_fit,
                                                ga2m_fid_mu, rho,
                                                thresh=cfg.student_min_fit)
    rec["students"]["ga2m"] = {
        "fidelity_mu": ga2m_fid_mu,
        "fidelity_nu": fidelity_metrics(Y_nu, ga2m.predict_logits(Z_nu),
                                        ds.y_test),
        "fidelity_fit": ga2m_fid_fit,
        "trained": ga2m_trained,
        "diagnosis": ga2m_note,
        "off_manifold_blowup": bool(ga2m_fid_mu["eta"] >= 1.0
                                    and ga2m_fid_fit["eta"] < 0.95),
        "eta_bootstrap": _bootstrap_eta(Y_mu, ga2m_mu, rng, cfg.n_bootstrap),
        "complexity": {**ga2m.complexity(), "interactions": ga2m_inter,
                       "budget_matched": do_match,
                       "target_components": int(target_components)},
        "total_effects": ga2m_te.tolist(),
        "agreement_vs_mc_sobol": rank_agreement(ga2m_te, teacher_total),
        "agreement_vs_permutation": rank_agreement(ga2m_te, perm),
        "agreement_vs_shap": (rank_agreement(ga2m_te, shap_res["importance"])
                              if shap_res["available"] else None),
    }

    # -- student-student agreement ----------------------------------------
    pairs = {}
    names = [n for n in ("gas_pce", "sdt", "ga2m")
             if "total_effects" in rec["students"].get(n, {})]
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a = np.array(rec["students"][names[i]]["total_effects"])
            b = np.array(rec["students"][names[j]]["total_effects"])
            pairs[f"{names[i]}__vs__{names[j]}"] = rank_agreement(a, b)
    rec["student_student_agreement"] = pairs

    tick("ga2m")
    rec["wall_clock_s"] = time.time() - t0
    rec["stage_seconds"] = {k: round(v, 2) for k, v in stage.items()}
    _write_record_atomic(path_json, rec)
    if verbose:
        g = rec["students"]["gas_pce"]
        print(f"  [{rec['cell']}] eta_mu={g['fidelity_mu']['eta']:.3f} "
              f"eta_fit={g['fidelity_fit']['eta']:.3f} "
              f"agree={g['fidelity_mu']['top1_agreement']:.3f} "
              f"comp={g['complexity']['active_components']} "
              f"({rec['wall_clock_s']:.0f}s)")
    return rec


def run_grid(cfg: RunConfig, dataset_keys=None, rhos=None, teachers=None,
             out_dir: str | None = None, overwrite: bool = False,
             order: str = "seed_major") -> list[dict]:
    """Run a grid of cells, resuming over completed ones.

    `order` decides what a half-finished run leaves you with, which matters
    because this grid takes more wall-clock than one Colab session.

      "seed_major" (default): seed outermost, then dataset, then rho. After
        the first pass every (dataset, rho) cell exists at seed 0, so every
        table in the report can be built -- at n=1, but complete -- and each
        later pass deepens it. It also puts rho innermost, which is what lets
        the teacher cache hit: the teacher does not depend on rho, so all four
        rhos of a (dataset, seed) reuse one training.

      "dataset_major": the original dataset -> teacher -> rho -> seed order.
        Finishes whole datasets at a time, so an interrupted run yields some
        datasets complete and others with nothing, and no cross-dataset table
        at all. It also defeats the teacher cache, because seed varies fastest
        and each (dataset, seed) teacher is evicted before rho returns to it.

    Failures are collected and reported at the end rather than only scrolling
    past, so "the grid finished" and "the grid produced every cell" are
    distinguishable.
    """
    cfg = _resolve_config(cfg)
    # A grid is the expensive thing to get wrong, so check up front that the
    # install can produce attributable numbers at all. A defaulted knob means
    # config.py is older than experiment.py, which is how one cell came to be
    # reported twice with different operating points.
    assert_consistent_install(cfg)
    # Print the source fingerprint once per grid. Two sessions that disagree
    # on a deterministic cell differ in their install, and without this line
    # that has to be inferred from the numbers instead of read off the log.
    _prov = run_provenance(cfg)
    print(f"[install] source={_prov['source_combined']} "
          f"config={_prov['config_digest']} rule={_prov['selection_rule']}"
          + (f" DEFAULTED={_prov['config_defaulted']}"
             if _prov.get("config_defaulted") else ""))
    dataset_keys = dataset_keys or [d.key for d in DATASETS_BY_KEY.values()]
    rhos = rhos if rhos is not None else [cfg.rho]
    teachers = teachers or [cfg.teacher]

    if order == "seed_major":
        plan = [(dk, tk, rho, seed) for seed in cfg.seeds
                for dk in dataset_keys for tk in teachers for rho in rhos]
    elif order == "dataset_major":
        plan = [(dk, tk, rho, seed) for dk in dataset_keys
                for tk in teachers for rho in rhos for seed in cfg.seeds]
    else:
        raise ValueError(f"unknown order {order!r}")

    recs, failures = [], []
    for dk, tk, rho, seed in plan:
        print(f"[run] {dk} teacher={tk} rho={rho} seed={seed}")
        try:
            recs.append(run_cell(dk, seed, cfg, rho=rho, teacher_kind=tk,
                                 out_dir=out_dir, overwrite=overwrite))
        except Exception as exc:
            print(f"  [FAIL] {dk}/{tk}/{rho}/{seed}: {exc}")
            failures.append((f"{dk}/{tk}/rho{rho}/seed{seed}", repr(exc)))

    print(f"\n[grid] {len(recs)}/{len(plan)} cells present, "
          f"{len(failures)} failed")
    if failures:
        print("[grid] failed cells (re-run these before building tables):")
        for cell, exc in failures:
            print(f"    {cell:40s} {exc[:90]}")
    return recs


def load_runs(out_dir: str, strict: bool = False) -> list[dict]:
    """Load every readable cell record, naming any that are not.

    One truncated file used to raise out of a list comprehension and take the
    analysis notebook down with a message that did not say which file. A
    damaged record is now reported and skipped, so the analysis runs on what
    is actually there; pass strict=True to raise instead.
    """
    p = Path(out_dir)
    runs, bad = [], []
    for f in sorted(p.glob("*.json")):
        rec = read_record(f)
        (runs.append(rec) if rec is not None else bad.append(f.name))
    if bad:
        msg = (f"{len(bad)} record(s) unreadable and skipped: "
               f"{', '.join(bad[:5])}{' ...' if len(bad) > 5 else ''}. "
               f"Re-run those cells (run_grid resumes and will redo them).")
        if strict:
            raise RuntimeError(msg)
        print(f"[load_runs] {msg}")
    # A grid is assembled over several sessions, so the records here need not
    # share a code version. Check before anything aggregates them: a table
    # whose rho column varies the code along with rho is worse than a missing
    # table, because it looks finished.
    if runs:
        audit_provenance(runs, strict=strict)
    return runs


### 1.12 LaTeX table emission

In [ ]:
"""Statistics for a multi-seed run.

Ten seeds only buy rigour if the analysis uses the structure the design
creates. Two facts drive everything here:

1. **Comparisons between students are paired.** Within one (dataset, seed)
   cell every student sees the *same* split, the *same* frozen teacher and the
   *same* query set. The seed-to-seed variation in task difficulty is therefore
   common to all three, and a paired test removes it. Comparing two
   `mean +/- sd` columns throws that away and is far less powerful at n=10.

2. **n=10 is small.** A normal-theory t-interval on ten values of a bounded,
   skewed quantity like `eta` is not credible. We use distribution-free
   procedures throughout: Wilcoxon signed-rank for paired differences, Cliff's
   delta for effect size, BCa bootstrap for interval estimates, and
   Clopper-Pearson for the certificate's hold rate (a proportion out of 10,
   where a Wald interval is actively misleading at p near 1).

Multiplicity is real: 4 datasets x 3 student pairs = 12 paired tests for a
single metric. `holm` adjusts within a declared family, and every reporting
function takes the family explicitly so the correction cannot silently be
applied to the wrong set.

Nothing here invents data: every function takes the per-seed values that the
run records already contain.
"""
from __future__ import annotations

from dataclasses import dataclass, asdict

import numpy as np

try:
    from scipy import stats as _sps
    HAS_SCIPY = True
except Exception:  # pragma: no cover
    HAS_SCIPY = False


# ---------------------------------------------------------------------------
# interval estimates
# ---------------------------------------------------------------------------


def bca_bootstrap_ci(x, alpha: float = 0.05, n_boot: int = 10000,
                     statistic=np.mean, seed: int = 0) -> dict:
    """Bias-corrected and accelerated bootstrap CI.

    BCa rather than the percentile interval because at n=10 the statistic's
    sampling distribution is usually both biased and skewed, and BCa corrects
    for each (z0 and the jackknife acceleration respectively). Falls back to
    the percentile interval if the acceleration is undefined.
    """
    x = np.asarray([v for v in np.ravel(x) if np.isfinite(v)], dtype=np.float64)
    n = x.size
    if n == 0:
        return {"point": float("nan"), "lo": float("nan"), "hi": float("nan"),
                "n": 0, "method": "none"}
    if n < 3:
        return {"point": float(statistic(x)), "lo": float(np.min(x)),
                "hi": float(np.max(x)), "n": n, "method": "range (n<3)"}

    rng = np.random.default_rng(seed)
    theta = float(statistic(x))
    boot = np.array([statistic(x[rng.integers(0, n, n)]) for _ in range(n_boot)])
    boot = boot[np.isfinite(boot)]
    if boot.size < 100:
        return {"point": theta, "lo": float(np.min(x)), "hi": float(np.max(x)),
                "n": n, "method": "range (bootstrap failed)"}

    prop = np.mean(boot < theta)
    method = "bca"
    if prop <= 0 or prop >= 1:
        lo, hi = np.percentile(boot, [100 * alpha / 2, 100 * (1 - alpha / 2)])
        return {"point": theta, "lo": float(lo), "hi": float(hi), "n": n,
                "method": "percentile (z0 undefined)"}

    from math import erf, sqrt
    ndtri = _sps.norm.ppf if HAS_SCIPY else _approx_ndtri
    ndtr = _sps.norm.cdf if HAS_SCIPY else (lambda z: 0.5 * (1 + erf(z / sqrt(2))))

    z0 = ndtri(prop)
    jack = np.array([statistic(np.delete(x, i)) for i in range(n)])
    jbar = jack.mean()
    num = ((jbar - jack) ** 3).sum()
    den = 6.0 * (((jbar - jack) ** 2).sum() ** 1.5)
    a = num / den if den != 0 else 0.0

    out = []
    for q in (alpha / 2, 1 - alpha / 2):
        zq = ndtri(q)
        adj = z0 + (z0 + zq) / max(1e-12, (1 - a * (z0 + zq)))
        out.append(float(np.percentile(boot, 100 * ndtr(adj))))
    lo, hi = sorted(out)
    return {"point": theta, "lo": lo, "hi": hi, "n": n, "method": method}


def _approx_ndtri(p):  # pragma: no cover - only without scipy
    from math import log, sqrt
    p = min(max(float(p), 1e-12), 1 - 1e-12)
    q = p - 0.5
    if abs(q) <= 0.425:
        r = 0.180625 - q * q
        return q * (((2509.08 * r + 33430.6) * r + 67265.8) * r + 45921.95) / \
               (((5226.5 * r + 28729.1) * r + 39307.9) * r + 21213.79)
    r = sqrt(-log(p if q < 0 else 1 - p))
    v = (2.32121 + r * (4.85014 + r * 2.32121)) / (1.0 + r * (3.75441 + r))
    return -v if q < 0 else v


def clopper_pearson(k: int, n: int, alpha: float = 0.05) -> dict:
    """Exact binomial CI for a hold rate out of n seeds.

    At k=n (the certificate holding on every seed) a Wald interval collapses to
    the single point 1.0, which overstates what ten seeds can establish. The
    exact interval reports the honest lower bound -- about 0.69 at 10/10.
    """
    if n <= 0:
        return {"rate": float("nan"), "lo": float("nan"), "hi": float("nan"), "n": 0}
    rate = k / n
    if not HAS_SCIPY:
        return {"rate": rate, "lo": float("nan"), "hi": float("nan"), "n": n}
    lo = 0.0 if k == 0 else float(_sps.beta.ppf(alpha / 2, k, n - k + 1))
    hi = 1.0 if k == n else float(_sps.beta.ppf(1 - alpha / 2, k + 1, n - k))
    return {"rate": rate, "lo": lo, "hi": hi, "n": n, "k": k}


# ---------------------------------------------------------------------------
# paired comparison
# ---------------------------------------------------------------------------


def cliffs_delta(a, b) -> float:
    """Non-parametric effect size in [-1, 1]: P(a>b) - P(a<b)."""
    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)
    a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
    if a.size == 0 or b.size == 0:
        return float("nan")
    diff = np.sign(a[:, None] - b[None, :])
    return float(diff.mean())


def cliffs_magnitude(delta: float) -> str:
    d = abs(delta)
    if not np.isfinite(d):
        return "n/a"
    return ("negligible" if d < 0.147 else "small" if d < 0.33
            else "medium" if d < 0.474 else "large")


@dataclass
class PairedResult:
    name_a: str
    name_b: str
    n_pairs: int
    median_diff: float          # median of (a - b) over seeds
    ci_lo: float
    ci_hi: float
    p_value: float
    p_adjusted: float
    cliffs_delta: float
    magnitude: str
    significant: bool

    def as_dict(self) -> dict:
        return asdict(self)


def paired_compare(a, b, name_a: str = "a", name_b: str = "b",
                   alpha: float = 0.05, seed: int = 0) -> PairedResult:
    """Wilcoxon signed-rank on the paired differences, with a BCa CI.

    `a[i]` and `b[i]` must come from the SAME seed; pairs where either value is
    missing are dropped. Wilcoxon rather than a paired t-test because ten
    differences cannot establish normality, and rather than the unpaired
    Mann-Whitney because discarding the pairing throws away the design.
    """
    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)
    n = min(a.size, b.size)
    a, b = a[:n], b[:n]
    ok = np.isfinite(a) & np.isfinite(b)
    a, b = a[ok], b[ok]
    diff = a - b

    if diff.size == 0:
        return PairedResult(name_a, name_b, 0, *([float("nan")] * 3),
                            float("nan"), float("nan"), "n/a", False)

    ci = bca_bootstrap_ci(diff, alpha=alpha, statistic=np.median, seed=seed)
    if HAS_SCIPY and diff.size >= 5 and np.any(diff != 0):
        try:
            p = float(_sps.wilcoxon(a, b, zero_method="wilcox").pvalue)
        except Exception:
            p = float("nan")
    else:
        p = float("nan")

    dlt = cliffs_delta(a, b)
    return PairedResult(name_a, name_b, int(diff.size),
                        float(np.median(diff)), ci["lo"], ci["hi"],
                        p, float("nan"), dlt, cliffs_magnitude(dlt), False)


def holm(results: list[PairedResult], alpha: float = 0.05) -> list[PairedResult]:
    """Holm-Bonferroni across a declared family of paired tests.

    Holm rather than plain Bonferroni (uniformly more powerful, same family-wise
    error guarantee) and rather than Benjamini-Hochberg because with a dozen
    tests backing individual claims in a paper we want family-wise control, not
    an expected false-discovery proportion.
    """
    idx = [i for i, r in enumerate(results) if np.isfinite(r.p_value)]
    m = len(idx)
    order = sorted(idx, key=lambda i: results[i].p_value)
    running = 0.0
    for rank, i in enumerate(order):
        adj = min(1.0, (m - rank) * results[i].p_value)
        running = max(running, adj)       # enforce monotonicity
        results[i].p_adjusted = running
        results[i].significant = bool(running <= alpha)
    for i, r in enumerate(results):
        if not np.isfinite(r.p_value):
            r.p_adjusted = float("nan")
            r.significant = False
    return results


# ---------------------------------------------------------------------------
# seed stability of an explanation
# ---------------------------------------------------------------------------


def rank_stability(vectors: list[np.ndarray]) -> dict:
    """How reproducible is the explanation itself across seeds?

    Mean pairwise Spearman between the per-seed total-effect vectors, plus the
    mean top-10 Jaccard. This is the direct measurement of failure mode (iv):
    an explanation that changes when only the seed changes is not a property of
    the teacher, however faithful the student looks.
    """
    V = [np.asarray(v, dtype=np.float64) for v in vectors
         if v is not None and np.isfinite(np.asarray(v, dtype=np.float64)).any()]
    if len(V) < 2:
        return {"mean_spearman": float("nan"), "mean_jaccard10": float("nan"),
                "n_seeds": len(V)}
    rhos, jacs = [], []
    for i in range(len(V)):
        for j in range(i + 1, len(V)):
            x, y = V[i], V[j]
            k = min(x.size, y.size)
            x, y = x[:k], y[:k]
            if HAS_SCIPY and not (np.allclose(x, x[0]) or np.allclose(y, y[0])):
                r = _sps.spearmanr(x, y).statistic
                if np.isfinite(r):
                    rhos.append(float(r))
            t = int(min(10, k))
            A = set(np.argsort(-x)[:t]); B = set(np.argsort(-y)[:t])
            jacs.append(len(A & B) / len(A | B))
    return {"mean_spearman": float(np.mean(rhos)) if rhos else float("nan"),
            "mean_jaccard10": float(np.mean(jacs)) if jacs else float("nan"),
            "n_seeds": len(V)}


# ---------------------------------------------------------------------------
# pulling per-seed values out of run records
# ---------------------------------------------------------------------------


def per_seed(runs, dataset: str, student: str, path: list[str],
             rho: float | None = None, teacher: str = "mlp") -> np.ndarray:
    """Per-seed values of one metric, ordered by seed so pairing is valid."""
    out = {}
    for r in runs:
        if r.get("dataset") != dataset or r.get("teacher_kind", "mlp") != teacher:
            continue
        if rho is not None and abs(r.get("rho", -1) - rho) > 1e-9:
            continue
        node = r.get("students", {}).get(student, {})
        for key in path:
            if not isinstance(node, dict) or key not in node:
                node = None
                break
            node = node[key]
        if node is not None:
            out[r["seed"]] = node
    return np.array([out[s] for s in sorted(out)], dtype=np.float64)


def per_seed_vectors(runs, dataset: str, student: str, key: str = "total_effects",
                     rho: float | None = None, teacher: str = "mlp") -> list:
    out = {}
    for r in runs:
        if r.get("dataset") != dataset or r.get("teacher_kind", "mlp") != teacher:
            continue
        if rho is not None and abs(r.get("rho", -1) - rho) > 1e-9:
            continue
        v = r.get("students", {}).get(student, {}).get(key)
        if v is not None:
            out[r["seed"]] = np.asarray(v, dtype=np.float64)
    return [out[s] for s in sorted(out)]


# ---------------------------------------------------------------------------
# the report the notebook prints and the tables consume
# ---------------------------------------------------------------------------


STUDENTS = ("gas_pce", "sdt", "ga2m")


def student_comparison_family(runs, datasets, metric_path=("fidelity_mu", "eta"),
                              rho=0.5, teacher="mlp", alpha=0.05,
                              lower_is_better=True) -> dict:
    """All paired student-vs-student tests for one metric, Holm-corrected
    together as one family (the correction spans datasets AND pairs)."""
    results, keys = [], []
    for d in datasets:
        vals = {s: per_seed(runs, d, s, list(metric_path), rho, teacher)
                for s in STUDENTS}
        for i in range(len(STUDENTS)):
            for j in range(i + 1, len(STUDENTS)):
                a, b = STUDENTS[i], STUDENTS[j]
                if vals[a].size == 0 or vals[b].size == 0:
                    continue
                results.append(paired_compare(vals[a], vals[b], a, b, alpha))
                keys.append((d, a, b))
    holm(results, alpha=alpha)
    return {
        "metric": ".".join(metric_path),
        "lower_is_better": lower_is_better,
        "family_size": len(results),
        "alpha": alpha,
        "tests": [{"dataset": k[0], **r.as_dict()} for k, r in zip(keys, results)],
    }


def certificate_audit(runs, datasets, rho=0.5, teacher="mlp") -> dict:
    """Hold rate with an exact binomial interval, plus tightness intervals.

    The denominator matters.  `certificate_report` writes None for its hold
    tests when eta >= 1, because Theorem 1 assumes eta < 1 and a test against
    an infinite bound passes for free; `per_seed` drops those, so the binomial
    is over seeds where the claim was actually testable.  That is the correct
    denominator but an invisible one, so `n_tested` and `n_violated` are
    reported beside it -- "6/6" and "6/6 of 10 cells" are different results
    and the table must not be able to show the first while meaning the second.
    """
    out = {}
    for d in datasets:
        holds = per_seed(runs, d, "gas_pce", ["certificate", "bound_holds"],
                         rho, teacher)
        tight = per_seed(runs, d, "gas_pce", ["certificate", "tightness"],
                         rho, teacher)
        eta = per_seed(runs, d, "gas_pce", ["certificate", "eta_mu"], rho, teacher)
        floor = per_seed(runs, d, "gas_pce", ["certificate", "mc_noise_floor"],
                         rho, teacher)
        delta = per_seed(runs, d, "gas_pce", ["certificate", "bound_delta"],
                         rho, teacher)
        viol = per_seed(runs, d, "gas_pce",
                        ["certificate", "precondition_violated"], rho, teacher)
        vac = per_seed(runs, d, "gas_pce", ["certificate", "vacuous"],
                       rho, teacher)
        k = int(np.nansum(holds)) if holds.size else 0
        out[d] = {
            "hold": clopper_pearson(k, int(holds.size)),
            "n_tested": int(holds.size),
            # eta >= 1: the theorem does not apply, so these are excluded from
            # the hold rate rather than counted as failures
            "n_violated": int(np.nansum(viol)) if viol.size else 0,
            "n_cells": int(eta.size),
            "tightness": bca_bootstrap_ci(tight),
            "eta_mu": bca_bootstrap_ci(eta),
            "floor_exceeds_delta": (
                float(np.mean(floor > delta)) if floor.size and delta.size
                else float("nan")),
            # sound-but-uninformative (2 <= Delta < inf), NOT the violated case
            "vacuous_rate": (float(np.nanmean(vac)) if vac.size
                             else float("nan")),
        }
    return out


def stability_audit(runs, datasets, rho=0.5, teacher="mlp") -> dict:
    return {d: {s: rank_stability(
                    per_seed_vectors(runs, d, s, "total_effects", rho, teacher))
                for s in STUDENTS}
            for d in datasets}


def summarise(runs, datasets, rho=0.5, teacher="mlp", alpha=0.05) -> dict:
    """Everything the analysis notebook needs, in one call."""
    return {
        "n_records": len(runs),
        "eta_mu_tests": student_comparison_family(
            runs, datasets, ("fidelity_mu", "eta"), rho, teacher, alpha),
        "agreement_tests": student_comparison_family(
            runs, datasets, ("agreement_vs_mc_sobol", "spearman"), rho, teacher,
            alpha, lower_is_better=False),
        "certificate": certificate_audit(runs, datasets, rho, teacher),
        "stability": stability_audit(runs, datasets, rho, teacher),
    }


def print_report(summary: dict) -> None:
    print(f"records: {summary['n_records']}\n")
    for title, fam in (("eta_mu (lower is better)", summary["eta_mu_tests"]),
                       ("Spearman vs teacher Sobol' (higher is better)",
                        summary["agreement_tests"])):
        print(f"=== paired student comparisons: {title} ===")
        print(f"    Wilcoxon signed-rank, Holm-corrected over "
              f"{fam['family_size']} tests, alpha={fam['alpha']}")
        print(f"    {'dataset':14s} {'pair':22s} {'med diff':>9s} "
              f"{'95% CI':>18s} {'p_adj':>8s} {'effect':>12s}")
        for t in fam["tests"]:
            star = "*" if t["significant"] else " "
            ci = f"[{t['ci_lo']:+.3f},{t['ci_hi']:+.3f}]"
            p = f"{t['p_adjusted']:.3f}" if np.isfinite(t["p_adjusted"]) else "  n/a"
            print(f"  {star} {t['dataset']:14s} {t['name_a']+' vs '+t['name_b']:22s} "
                  f"{t['median_diff']:+9.3f} {ci:>18s} {p:>8s} "
                  f"{t['magnitude']:>12s}")
        print()

    print("=== certificate ===")
    print(f"    {'dataset':14s} {'holds':>14s} {'tightness (95% CI)':>26s} "
          f"{'vacuous':>8s} {'floor>D':>8s}")
    for d, c in summary["certificate"].items():
        h = c["hold"]
        hs = (f"{h.get('k',0)}/{h['n']} [{h['lo']:.2f},{h['hi']:.2f}]"
              if np.isfinite(h.get("lo", np.nan)) else f"{h.get('k',0)}/{h['n']}")
        t = c["tightness"]
        ts = f"{t['point']:.3f} [{t['lo']:.3f},{t['hi']:.3f}]"
        print(f"    {d:14s} {hs:>14s} {ts:>26s} "
              f"{c['vacuous_rate']:>8.2f} {c['floor_exceeds_delta']:>8.2f}")
    print()

    print("=== explanation stability across seeds (mean pairwise) ===")
    print(f"    {'dataset':14s} " + " ".join(f"{s:>22s}" for s in STUDENTS))
    for d, per in summary["stability"].items():
        cells = []
        for s in STUDENTS:
            v = per[s]
            cells.append(f"rho={v['mean_spearman']:.2f} J={v['mean_jaccard10']:.2f}"
                         if np.isfinite(v["mean_spearman"]) else "n/a")
        print(f"    {d:14s} " + " ".join(f"{c:>22s}" for c in cells))


In [ ]:
"""Emit the LaTeX tables that paper/main.tex \\inputs.

Each function writes a complete `table` environment, caption and label
included, so main.tex only ever has to \\input the file.  Nothing here invents
a number: every cell is mean +/- sd over whatever seeds actually completed, and
cells with no completed runs print an em-dash.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np

STUDENT_LABEL = {"gas_pce": r"\GASPCE\ (ours)", "sdt": r"\SDT", "ga2m": r"\GAM"}
DATASET_LABEL = {"bank": "bank-marketing", "credit": "credit-default",
                 "gas": "gas-drift", "musk": "musk v2"}


def _ms(vals, fmt="{:.3f}") -> str:
    v = np.array([x for x in vals if x is not None and np.isfinite(x)],
                 dtype=np.float64)
    if v.size == 0:
        return "---"
    if v.size == 1:
        return fmt.format(v[0])
    return (fmt + r"\,$\pm$\,"+ fmt.replace("{:", "{:")).format(v.mean(), v.std(ddof=1))


def _collect(runs, dataset, student, path, rho=None, teacher="mlp"):
    out = []
    for r in runs:
        if r["dataset"] != dataset or r.get("teacher_kind", "mlp") != teacher:
            continue
        if rho is not None and abs(r["rho"] - rho) > 1e-9:
            continue
        node = r["students"].get(student, {})
        for key in path:
            if not isinstance(node, dict) or key not in node:
                node = None
                break
            node = node[key]
        if node is not None:
            out.append(node)
    return out


def _header(ncol_spec: str, caption: str, label: str,
            fit: bool = False) -> list[str]:
    """`fit=True` wraps the tabular in \\resizebox so a wide table cannot
    overflow the text block -- the rho-ablation table is 9 columns and does
    exactly that at \\footnotesize."""
    L = [r"\begin{table}[t]", r"\caption{" + caption + "}",
         r"\label{" + label + "}", r"\begin{center}\footnotesize"]
    if fit:
        L.append(r"\resizebox{\linewidth}{!}{%")
    L += [r"\begin{tabular}{" + ncol_spec + "}", r"\toprule"]
    return L


def _footer(fit: bool = False) -> list[str]:
    L = [r"\bottomrule", r"\end{tabular}"]
    if fit:
        L.append("}")
    return L + [r"\end{center}", r"\end{table}"]


def table_fidelity(runs, out_path, rho=0.5, teacher="mlp") -> str:
    datasets = [d for d in DATASET_LABEL if any(r["dataset"] == d for r in runs)]
    L = _header("@{}llccccc@{}",
                "Teacher--student fidelity at the operating point "
                f"($\\rho={rho:g}$, {teacher.upper()} teacher), mean $\\pm$ s.d. "
                "over seeds. $\\eta_\\mu$ is the certified quantity; "
                "$\\eta_\\nu$ and top-1 agreement are the prediction-level "
                "numbers the literature reports.",
                "tab:fidelity")
    L.append(r"\textbf{Data} & \textbf{Student} & $\eta_\mu\downarrow$ & "
             r"$\eta_\nu\downarrow$ & \textbf{top-1 agr.}$\uparrow$ & "
             r"\textbf{JS}$\downarrow$ & \textbf{components} \\")
    L.append(r"\midrule")
    for di, d in enumerate(datasets):
        for si, s in enumerate(("gas_pce", "sdt", "ga2m")):
            cells = [
                _ms(_collect(runs, d, s, ["fidelity_mu", "eta"], rho, teacher)),
                _ms(_collect(runs, d, s, ["fidelity_nu", "eta"], rho, teacher)),
                _ms(_collect(runs, d, s, ["fidelity_mu", "top1_agreement"], rho, teacher)),
                _ms(_collect(runs, d, s, ["fidelity_mu", "js_divergence"], rho, teacher)),
                _ms(_collect(runs, d, s, ["complexity", "active_components"], rho, teacher)
                    or _collect(runs, d, s, ["complexity", "components"], rho, teacher)
                    or _collect(runs, d, s, ["complexity", "internal_nodes"], rho, teacher),
                    "{:.0f}"),
            ]
            name = DATASET_LABEL[d] if si == 0 else ""
            L.append(f"{name} & {STUDENT_LABEL[s]} & " + " & ".join(cells) + r" \\")
        if di < len(datasets) - 1:
            L.append(r"\midrule")
    L += _footer()
    return _write(L, out_path)


def table_explanation(runs, out_path, rho=0.5, teacher="mlp") -> str:
    datasets = [d for d in DATASET_LABEL if any(r["dataset"] == d for r in runs)]
    L = _header("@{}llcccc@{}",
                "Explanation agreement: each student's total-effect vector "
                "against the teacher's Monte-Carlo total Sobol' indices and "
                "against global KernelSHAP on the teacher. Spearman $\\rho$ and "
                "top-10 Jaccard, mean $\\pm$ s.d. over seeds.",
                "tab:explanation")
    L.append(r"\textbf{Data} & \textbf{Student} & $\rho_s$ vs.\ Sobol' & "
             r"Jac$_{10}$ vs.\ Sobol' & $\rho_s$ vs.\ SHAP & "
             r"Jac$_{10}$ vs.\ SHAP \\")
    L.append(r"\midrule")
    for di, d in enumerate(datasets):
        for si, s in enumerate(("gas_pce", "sdt", "ga2m")):
            cells = [
                _ms(_collect(runs, d, s, ["agreement_vs_mc_sobol", "spearman"], rho, teacher)),
                _ms(_collect(runs, d, s, ["agreement_vs_mc_sobol", "jaccard_top10"], rho, teacher)),
                _ms(_collect(runs, d, s, ["agreement_vs_shap", "spearman"], rho, teacher)),
                _ms(_collect(runs, d, s, ["agreement_vs_shap", "jaccard_top10"], rho, teacher)),
            ]
            name = DATASET_LABEL[d] if si == 0 else ""
            L.append(f"{name} & {STUDENT_LABEL[s]} & " + " & ".join(cells) + r" \\")
        if di < len(datasets) - 1:
            L.append(r"\midrule")
    L += _footer()
    return _write(L, out_path)


def table_certificate(runs, out_path, rho=0.5, teacher="mlp") -> str:
    datasets = [d for d in DATASET_LABEL if any(r["dataset"] == d for r in runs)]
    L = _header("@{}lcccccc@{}",
                "The certificate in practice (\\GASPCE). "
                "$\\Delta(\\hat\\eta_\\mu)=4\\hat\\eta_\\mu/(1-\\hat\\eta_\\mu)$ "
                "is the guaranteed bound and ``measured'' is "
                "$\\max_j|T_j(\\fS)-T_j(\\fT)|$, the quantity in the second "
                "half of Eq.~\\ref{eq:cert}. The teacher's indices are "
                "themselves Monte-Carlo estimates, so the comparison cannot "
                "resolve below the reference estimator's own sampling error: "
                "``floor'' is that error, measured by estimating the teacher's "
                "indices twice from independent samples, and ``holds'' tests "
                "measured $\\le\\Delta+$floor. Where floor~$>\\Delta$ the "
                "experiment is limited by the reference, not by the bound. "
                "Seeds with $\\hat\\eta_\\mu\\ge1$ are excluded from "
                "``holds'' rather than scored: Theorem~\\ref{thm:cert} assumes "
                "$\\eta<1$, so there is no claim to test, and a comparison "
                "against an infinite bound would pass for free. Any such "
                "seeds are counted beneath the table.",
                "tab:certificate")
    L.append(r"\textbf{Data} & $\hat\eta_\mu$ & $\Delta(\hat\eta_\mu)$ & "
             r"\textbf{floor} & \textbf{measured} & \textbf{tightness} & "
             r"\textbf{holds} \\")
    L.append(r"\midrule")
    n_viol = 0
    for d in datasets:
        c = lambda k, f="{:.3f}": _ms(
            _collect(runs, d, "gas_pce", ["certificate", k], rho, teacher), f)
        n_viol += sum(bool(v) for v in _collect(
            runs, d, "gas_pce", ["certificate", "precondition_violated"],
            rho, teacher))
        L.append(" & ".join([DATASET_LABEL[d], c("eta_mu"), c("bound_delta"),
                             c("mc_noise_floor"), c("measured_max_total"),
                             c("tightness"),
                             c("bound_holds", "{:.2f}")]) + r" \\")
    if n_viol:
        L.append(r"\midrule")
        L.append(r"\multicolumn{7}{@{}l@{}}{\footnotesize "
                 + f"{n_viol} seed-cell(s) had $\\hat\\eta_\\mu\\ge1$ and are "
                 + r"excluded from ``holds'' (Theorem~\ref{thm:cert} does not "
                 + r"apply).} \\")
    L += _footer()
    return _write(L, out_path)


def table_hparams(runs, out_path) -> str:
    cfg = runs[0]["config"] if runs else {}
    rows = [("datasets", "bank-marketing, credit-default, gas-drift, musk v2"),
            ("seeds", str(len(cfg.get("seeds", [])))),
            ("teacher epochs", str(cfg.get("epochs", "---"))),
            ("teacher hidden", str(cfg.get("hidden", "---"))),
            ("PCE degree $p$ / $q$-norm",
             f"{cfg.get('degree','---')} / {cfg.get('q_norm','---')}"),
            ("screening $m$", str(cfg.get("screen_m", "---"))),
            ("interaction penalty $\\gamma$", str(cfg.get("gamma", "---"))),
            ("$\\lambda$ path length", str(cfg.get("n_lambda", "---"))),
            ("query size $|Z|$ / $\\rho$",
             f"{cfg.get('n_query','---')} / {cfg.get('rho','---')}"),
            ("MC Sobol' base $N$", str(cfg.get("sobol_mc_base", "---"))),
            ("KernelSHAP explain / background",
             f"{cfg.get('shap_n_explain','---')} / {cfg.get('shap_n_background','---')}")]
    # surface any knob that was defaulted because config.py was stale, so the
    # table cannot present a defaulted value as a deliberate one
    defaulted = sorted({k for r in runs
                        for k in (r.get("config_defaulted") or [])})
    if defaulted:
        rows.append(("\\textbf{defaulted (stale config.py)}",
                     "\\texttt{" + ", ".join(
                         d.replace("_", r"\_") for d in defaulted[:6])
                     + (", \\dots" if len(defaulted) > 6 else "") + "}"))
    wall = {}
    for r in runs:
        wall.setdefault(r["dataset"], []).append(r.get("wall_clock_s", np.nan))
    for k, v in wall.items():
        rows.append((f"wall-clock {DATASET_LABEL.get(k,k)} (s/cell)",
                     _ms(v, "{:.0f}")))

    L = _header("@{}ll@{}", "Full configuration and measured wall-clock, "
                "emitted directly from the run records.", "tab:hparams")
    L.append(r"\textbf{Setting} & \textbf{Value} \\")
    L.append(r"\midrule")
    for k, v in rows:
        L.append(f"{k} & {v} " + r"\\")
    L += _footer()
    return _write(L, out_path)


def table_ablation_rho(runs, out_path, rhos=(0.0, 0.25, 0.5, 1.0)) -> str:
    datasets = [d for d in DATASET_LABEL if any(r["dataset"] == d for r in runs)]
    L = _header("@{}l" + "cc" * len(rhos) + "@{}",
                "Ablation over the product-measure query fraction $\\rho$ "
                "(\\GASPCE). The $\\eta_\\mu - \\eta_\\nu$ gap at $\\rho=0$ is "
                "Proposition~\\ref{prop:measure} in action: a student fitted "
                "only on the data manifold has no certificate. Markers give "
                "the standing of the bound, which is three-valued. "
                "$^{\\ddagger}$: $1/3\\le\\eta_\\mu<1$, so $\\Delta\\ge2$ and "
                "the bound, while sound, asserts no more than the $[0,1]$ "
                "range of a normalised index. $^{\\dagger}$: "
                "$\\eta_\\mu\\ge1$, so the hypothesis of "
                "Theorem~\\ref{thm:cert} fails and $\\Delta$ is undefined, not "
                "merely large -- the student is no better under $\\mu$ than "
                "predicting the teacher's mean, and its analytic indices carry "
                "no established relation to the teacher even though the same "
                "student fits the data measure well. Daggered cells are not "
                "optimisation failures and are excluded from the hold rate in "
                "Table~\\ref{tab:certificate} rather than scored; they are the "
                "strongest form of the reference-measure problem.",
                "tab:ablation-rho", fit=True)
    head = r"\textbf{Data}"
    for r_ in rhos:
        head += rf" & \multicolumn{{2}}{{c}}{{$\rho={r_:g}$}}"
    L.append(head + r" \\")
    L.append(r"\cmidrule(lr){2-" + str(1 + 2 * len(rhos)) + "}")
    L.append(" & " + " & ".join([r"$\eta_\mu$", r"$\eta_\nu$"] * len(rhos)) + r" \\")
    L.append(r"\midrule")
    for d in datasets:
        cells = []
        for r_ in rhos:
            mu = _collect(runs, d, "gas_pce", ["fidelity_mu", "eta"], r_)
            nu = _collect(runs, d, "gas_pce", ["fidelity_nu", "eta"], r_)
            cell = _ms(mu)
            # three-valued marking, matching metrics.certificate_report:
            # dagger   = precondition violated (eta >= 1), no bound exists
            # ddagger  = sound but vacuous (Delta >= 2, i.e. eta >= 1/3)
            # unmarked = informative
            vals = [v for v in mu if v is not None and np.isfinite(v)]
            if vals:
                m = float(np.mean(vals))
                if m >= 1.0:
                    cell += r"$^{\dagger}$"
                elif m >= 1.0 / 3.0:
                    cell += r"$^{\ddagger}$"
            cells.append(cell)
            cells.append(_ms(nu))
        L.append(DATASET_LABEL[d] + " & " + " & ".join(cells) + r" \\")
    L += _footer(fit=True)
    return _write(L, out_path)


def _write(lines: list[str], out_path) -> str:
    txt = "\n".join(lines) + "\n"
    p = Path(out_path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(txt)
    return txt


def emit_all_tables(runs, results_dir="paper/results", rho=0.5, teacher="mlp"):
    d = Path(results_dir)
    table_fidelity(runs, d / "tab_fidelity.tex", rho, teacher)
    table_explanation(runs, d / "tab_explanation.tex", rho, teacher)
    table_certificate(runs, d / "tab_certificate.tex", rho, teacher)
    table_hparams(runs, d / "tab_hparams.tex")
    table_ablation_rho(runs, d / "tab_ablation_rho.tex")
    # multi-seed statistics: only meaningful once several seeds are present
    n_seeds = len({(r.get("dataset"), r.get("seed")) for r in runs})
    if n_seeds >= 3:
        table_paired_tests(runs, d / "tab_paired.tex", rho=rho, teacher=teacher)
        table_certificate_ci(runs, d / "tab_certci.tex", rho=rho, teacher=teacher)
        table_stability(runs, d / "tab_stability.tex", rho=rho, teacher=teacher)
    return sorted(str(p) for p in d.glob("*.tex"))


# ---------------------------------------------------------------------------
# multi-seed statistical tables (see pact/stats.py)
# ---------------------------------------------------------------------------


def table_paired_tests(runs, out_path, datasets=None, rho=0.5, teacher="mlp",
                       alpha=0.05) -> str:
    """Paired student-vs-student tests on eta_mu, Holm-corrected.

    Replaces eyeballing two `mean +/- sd` columns. Within a (dataset, seed)
    cell all students see the same teacher and query set, so the comparison is
    paired and the seed-to-seed difficulty variation cancels.
    """
    from . import stats as S
    datasets = datasets or [d for d in DATASET_LABEL
                            if any(r["dataset"] == d for r in runs)]
    fam = student_comparison_family(runs, datasets, ("fidelity_mu", "eta"),
                                      rho, teacher, alpha)
    L = _header("@{}llccccc@{}",
                "Paired student comparisons on $\\eta_\\mu$ (lower is better). "
                "Within each (dataset, seed) cell all students see the same "
                "frozen teacher and the same query set, so differences are "
                "paired and seed-to-seed task difficulty cancels. Wilcoxon "
                "signed-rank, Holm-corrected over the "
                f"{fam['family_size']} tests in this table; $\\star$ marks "
                f"$p_{{\\mathrm{{adj}}}}\\le{alpha}$. Effect size is Cliff's "
                "$\\delta$. A negative median difference favours the first "
                "student.", "tab:paired")
    L.append(r"\textbf{Data} & \textbf{Comparison} & $n$ & "
             r"\textbf{median }$\Delta\eta_\mu$ & \textbf{95\% CI} & "
             r"$p_{\mathrm{adj}}$ & \textbf{effect} \\")
    L.append(r"\midrule")
    last = None
    for t in fam["tests"]:
        nm = DATASET_LABEL.get(t["dataset"], t["dataset"])
        show = nm if t["dataset"] != last else ""
        last = t["dataset"]
        star = r"$\star$" if t["significant"] else ""
        pair = (f"{STUDENT_LABEL.get(t['name_a'], t['name_a'])} vs.\\ "
                f"{STUDENT_LABEL.get(t['name_b'], t['name_b'])}")
        p = (f"{t['p_adjusted']:.3f}" if np.isfinite(t["p_adjusted"]) else "---")
        ci = (f"$[{t['ci_lo']:+.3f},{t['ci_hi']:+.3f}]$"
              if np.isfinite(t["ci_lo"]) else "---")
        L.append(f"{show} & {pair} & {t['n_pairs']} & "
                 f"${t['median_diff']:+.3f}$ & {ci} & {p}\\,{star} & "
                 f"{t['magnitude']} " + r"\\")
    L += _footer()
    return _write(L, out_path)


def table_certificate_ci(runs, out_path, datasets=None, rho=0.5,
                         teacher="mlp") -> str:
    """Certificate audit with an exact binomial hold rate and BCa intervals."""
    from . import stats as S
    datasets = datasets or [d for d in DATASET_LABEL
                            if any(r["dataset"] == d for r in runs)]
    aud = certificate_audit(runs, datasets, rho, teacher)
    L = _header("@{}lccccc@{}",
                "Certificate audit over seeds. ``Holds'' is the exact "
                "(Clopper--Pearson) 95\\% interval for the hold rate: ten "
                "successes out of ten establish a rate above about $0.69$, "
                "not $1.00$, and reporting $1.00\\pm0.00$ would overstate "
                "what ten seeds can show. Tightness and $\\hat\\eta_\\mu$ "
                "carry BCa bootstrap intervals over seeds. ``Vacuous'' is the "
                "fraction of \\emph{testable} seeds with $2\\le\\Delta<\\infty$ "
                "-- the bound is sound but asserts no more than the $[0,1]$ "
                "range of a normalised index -- and ``floor$>\\Delta$'' "
                "the fraction where the Monte-Carlo reference, not the bound, "
                "limits the comparison. Seeds with $\\hat\\eta_\\mu\\ge1$ are "
                "a separate state: the theorem's hypothesis fails, so they are "
                "neither tested nor counted vacuous, and the ``holds'' "
                "denominator shows the testable count against the total.",
                "tab:certci")
    L.append(r"\textbf{Data} & \textbf{holds (exact 95\% CI)} & "
             r"$\hat\eta_\mu$ \textbf{(95\% CI)} & "
             r"\textbf{tightness (95\% CI)} & \textbf{vacuous} & "
             r"\textbf{floor}$>\Delta$ \\")
    L.append(r"\midrule")
    for d in datasets:
        c = aud[d]
        h = c["hold"]
        # the denominator is the testable count; show it against the total
        # whenever a seed was dropped for eta >= 1, so "6/6" cannot be read
        # as "6 of every cell run"
        den = (f"{h['n']}" if c.get("n_violated", 0) == 0
               else f"{h['n']}" + r"\,\textsuperscript{of~" + f"{c['n_cells']}" + "}")
        hs = (f"{h.get('k',0)}/{den} $[{h['lo']:.2f},{h['hi']:.2f}]$"
              if np.isfinite(h.get("lo", np.nan))
              else f"{h.get('k',0)}/{den}")
        def ci(k):
            v = c[k]
            return (f"{v['point']:.3f} $[{v['lo']:.3f},{v['hi']:.3f}]$"
                    if np.isfinite(v["point"]) else "---")
        L.append(" & ".join([DATASET_LABEL.get(d, d), hs, ci("eta_mu"),
                             ci("tightness"),
                             f"{c['vacuous_rate']:.2f}",
                             f"{c['floor_exceeds_delta']:.2f}"]) + r" \\")
    L += _footer()
    return _write(L, out_path)


def table_stability(runs, out_path, datasets=None, rho=0.5,
                    teacher="mlp") -> str:
    """Is the explanation itself reproducible when only the seed changes?"""
    from . import stats as S
    datasets = datasets or [d for d in DATASET_LABEL
                            if any(r["dataset"] == d for r in runs)]
    aud = stability_audit(runs, datasets, rho, teacher)
    L = _header("@{}l" + "cc" * len(STUDENTS) + "@{}",
                "Seed stability of the explanation: mean pairwise Spearman "
                "$\\rho_s$ and top-10 Jaccard between the per-seed "
                "total-effect vectors of one student on one dataset. Nothing "
                "but the seed changes between the vectors being compared, so "
                "this upper-bounds how much of any explanation is a property "
                "of the teacher rather than of the fit. Low $\\rho_s$ with "
                "high Jaccard is the signature of failure mode (iv), "
                "attribution splitting among correlated features.",
                "tab:stability", fit=True)
    head = r"\textbf{Data}"
    for s in STUDENTS:
        head += rf" & \multicolumn{{2}}{{c}}{{{STUDENT_LABEL.get(s, s)}}}"
    L.append(head + r" \\")
    L.append(r"\cmidrule(lr){2-" + str(1 + 2 * len(STUDENTS)) + "}")
    L.append(" & " + " & ".join([r"$\rho_s$", r"Jac$_{10}$"] * len(STUDENTS))
             + r" \\")
    L.append(r"\midrule")
    for d in datasets:
        cells = []
        for s in STUDENTS:
            v = aud[d][s]
            cells.append(f"{v['mean_spearman']:.2f}"
                         if np.isfinite(v["mean_spearman"]) else "---")
            cells.append(f"{v['mean_jaccard10']:.2f}"
                         if np.isfinite(v["mean_jaccard10"]) else "---")
        L.append(DATASET_LABEL.get(d, d) + " & " + " & ".join(cells) + r" \\")
    L += _footer(fit=True)
    return _write(L, out_path)


### 1.13 Figures

In [ ]:
"""Figures for the report. Matplotlib only, no seaborn, no styling magic."""
from __future__ import annotations

from pathlib import Path

import numpy as np

DATASET_LABEL = {"bank": "bank-marketing", "credit": "credit-default",
                 "gas": "gas-drift", "musk": "musk v2"}
COLORS = {"bank": "#2F4858", "credit": "#33658A", "gas": "#F6AE2D",
          "musk": "#C1352B"}


def _bound(eta):
    eta = np.asarray(eta, dtype=np.float64)
    out = np.where(eta < 1.0, 4.0 * eta / np.maximum(1.0 - eta, 1e-9), np.inf)
    return out


def frontier_figure(runs, out_path="figures/frontier.pdf", rho=0.5,
                    teacher="mlp", clip=2.0):
    """Certified explanation frontier: active components vs. Delta(eta_mu)."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(4.2, 3.1))
    plotted = 0
    for key in DATASET_LABEL:
        curves = [r["pce_frontier"] for r in runs
                  if r["dataset"] == key and abs(r["rho"] - rho) < 1e-9
                  and r.get("teacher_kind", "mlp") == teacher
                  and r.get("pce_frontier")]
        if not curves:
            continue
        n = min(len(c) for c in curves)
        comp = np.median([[p["active_components"] for p in c[:n]] for c in curves],
                         axis=0)
        eta = np.median([[p["eta_mu"] for p in c[:n]] for c in curves], axis=0)
        delta = np.clip(_bound(eta), None, clip)
        order = np.argsort(comp)
        ax.plot(comp[order], delta[order], marker="o", ms=3, lw=1.2,
                color=COLORS[key], label=DATASET_LABEL[key])
        plotted += 1

    ax.axhline(2.0, ls="--", lw=0.8, color="0.6")
    ax.text(0.98, 2.02, "vacuous bound", ha="right", va="bottom",
            fontsize=7, color="0.45", transform=ax.get_yaxis_transform())
    ax.set_xscale("log")
    ax.set_xlabel("active HDMR components $|\\mathcal{U}|$")
    ax.set_ylabel(r"certified error $\Delta(\hat\eta_\mu)$")
    ax.set_title("Certified explanation frontier", fontsize=9)
    if plotted:
        ax.legend(fontsize=7, frameon=False)
    fig.tight_layout()
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    return out_path


def agreement_vs_eta_figure(runs, out_path="figures/agreement_vs_eta.pdf"):
    """The OP1 decoupling test: does label agreement predict eta_mu?"""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from scipy.stats import spearmanr

    fig, ax = plt.subplots(figsize=(4.2, 3.1))
    xs, ys = [], []
    for key in DATASET_LABEL:
        px, py = [], []
        for r in runs:
            if r["dataset"] != key:
                continue
            for sname, s in r["students"].items():
                fm = s.get("fidelity_mu")
                if not fm:
                    continue
                if np.isfinite(fm["eta"]) and np.isfinite(fm["top1_agreement"]):
                    px.append(fm["top1_agreement"])
                    py.append(fm["eta"])
        if px:
            ax.scatter(px, py, s=12, alpha=0.65, color=COLORS[key],
                       label=DATASET_LABEL[key], edgecolors="none")
            xs += px
            ys += py

    if len(xs) > 3:
        rho_s = spearmanr(xs, ys).statistic
        ax.set_title(f"Label agreement vs. certified residual "
                     f"(Spearman $\\rho$ = {rho_s:.2f})", fontsize=8)
    else:
        ax.set_title("Label agreement vs. certified residual", fontsize=9)
    ax.set_xlabel("top-1 agreement with teacher")
    ax.set_ylabel(r"$\eta_\mu$")
    ax.legend(fontsize=7, frameon=False)
    fig.tight_layout()
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    return out_path


def emit_all_figures(runs, figures_dir="figures", rho=0.5, teacher="mlp"):
    return [frontier_figure(runs, f"{figures_dir}/frontier.pdf", rho, teacher),
            agreement_vs_eta_figure(runs, f"{figures_dir}/agreement_vs_eta.pdf")]


In [ ]:
import types
_defined = [k for k, v in list(globals().items())
            if isinstance(v, (type, types.FunctionType))
            and not k.startswith('_')]
print(f'implementation loaded: {len(_defined)} classes and functions')
print('table emitter :', emit_all_tables.__name__)
print('figure emitter:', emit_all_figures.__name__)

## 2. Self-test — verify Theorem 1 before touching real data

The certificate is a theorem, so it cannot fail on correct code. This cell is the guard-rail that proves the implementation matches the proof: it recovers a known HDMR decomposition exactly, then checks the bound against randomly perturbed students.

In [ ]:
# Verify Theorem 1 before trusting anything downstream.
# Perturb a known student at several magnitudes and check the certificate.
rng_t = np.random.default_rng(0)

Z_t = rng_t.random((20000, 6))
def _f_known(A):
    p = [legendre_1d(A[:, j], 1)[:, 1] for j in range(A.shape[1])]
    return (2.0 * p[0] + 1.0 * p[1] + 0.5 * p[0] * p[2])[:, None]

pce_t = GasPCE(d=6, degree=4, q_norm=0.9, screen_m=6, n_lambda=6,
               lambda_min_ratio=1e-6, max_iter=600).fit(Z_t, _f_known(Z_t))
fit_t = pce_t.path[-1]

print("recovered HDMR components (true: f_0 alone, f_1 alone, f_0 x f_2):")
for u, s in sorted(fit_t.sobol.items(), key=lambda kv: -kv[1]):
    if s > 1e-3:
        print(f"   {str(u):10s} S_u = {s:.4f}   (exact: "
              f"{ {(0,):4/5.25, (1,):1/5.25, (0,2):0.25/5.25}.get(u, 0):.4f})")

Z_mu_t = product_measure_sample(Z_t, 20000, rng_t)
Y_mu_t = _f_known(Z_mu_t)
true_spec = {k: v for k, v in fit_t.sobol.items() if v > 1e-12}
Psi_t = design_matrix(Z_mu_t, pce_t.index_set)

print("\n  scale      eta     bound Delta(eta)   measured L1   holds")
worst = 0.0
for scale in (0.01, 0.05, 0.10, 0.20, 0.30):
    pert = fit_t.coef + scale * np.abs(fit_t.coef).max() * \
           rng_t.standard_normal(fit_t.coef.shape)
    eta = relative_l2(Y_mu_t, Psi_t @ pert)
    spec = {}
    for u, g in pce_t.index_set.groups().items():
        e = float((pert[g] ** 2).sum())
        if e > 0:
            spec[u] = e
    tot = sum(spec.values())
    spec = {u: e / tot for u, e in spec.items()}
    l1 = sobol_spectrum_l1(true_spec, spec)
    bound = certificate_bound(eta)
    worst = max(worst, l1 / bound if np.isfinite(bound) and bound > 0 else 0.0)
    print(f"  {scale:5.2f}   {eta:7.4f}   {bound:14.4f}   {l1:11.4f}   "
          f"{'OK' if l1 <= bound + 1e-9 else 'VIOLATED'}")

assert worst <= 1.0, "Theorem 1 violated — this is an implementation bug"
print(f"\nworst measured/bound ratio = {worst:.3f}  ->  certificate holds")

## 3. One cell end to end

Run a single `(bank, seed 0)` cell first. If this works, the grid will. The number to watch is `holds=True`.

In [ ]:
cfg = RunConfig()
QUICK = True   #@param {type:"boolean"}   <- set False for the real run
if QUICK:
    cfg = cfg.to_quick()
cfg.out_dir = OUT_DIR

print(f"seeds  : {list(cfg.seeds)}")
print(f"epochs : {cfg.epochs}")
print(f"PCE    : p={cfg.degree}, q={cfg.q_norm}, m={cfg.screen_m}, "
      f"gamma={cfg.gamma}, |lambda path|={cfg.n_lambda}\n")

rec = run_cell("bank", seed=0, cfg=cfg, rho=cfg.rho, teacher_kind="mlp",
               out_dir=OUT_DIR)

g = rec["students"]["gas_pce"]
c = g["certificate"]
print()
print(f"teacher test acc : {rec['teacher']['acc_test']:.4f}")
print(f"eta_mu / eta_nu  : {g['fidelity_mu']['eta']:.4f} / {g['fidelity_nu']['eta']:.4f}")
print(f"top-1 agreement  : {g['fidelity_mu']['top1_agreement']:.4f}")
print(f"components       : {g['complexity']['active_components']} "
      f"of basis {g['complexity']['basis_size']}")
print(f"certificate      : Delta={c['bound_delta']:.4f}  "
      f"measured={c['measured_l1']:.4f}  holds={c['bound_holds']}")
print(f"vs teacher Sobol : Spearman {g['agreement_vs_mc_sobol']['spearman']:.3f}")

## 4. The grid

### 4.1 Main grid — 4 datasets × 10 seeds

This is the run behind Tables 2–4. Completed cells are skipped, so re-running after a disconnect is cheap.

In [ ]:
main_runs = run_grid(cfg, dataset_keys=[d.key for d in DATASETS],
                     rhos=[cfg.rho], teachers=["mlp"], out_dir=OUT_DIR)
print(f"\ncompleted this pass: {len(main_runs)}")

### 4.2 Ablations — ρ and the teacher family

ρ = 0 fits the student only on the data manifold, where Proposition 3 says the certificate is void even if on-manifold fidelity looks excellent. The GBM teacher is included because the capacity argument for deep models on tabular data is contested.

In [ ]:
ABLATE_ALL = False  #@param {type:"boolean"}
abl_keys = [d.key for d in DATASETS] if ABLATE_ALL else ["bank", "credit"]

rho_runs = run_grid(cfg, dataset_keys=abl_keys,
                    rhos=[r for r in RHO_ABLATION if abs(r - cfg.rho) > 1e-9],
                    teachers=["mlp"], out_dir=OUT_DIR)
gbm_runs = run_grid(cfg, dataset_keys=["bank", "credit"], rhos=[cfg.rho],
                    teachers=["gbm"], out_dir=OUT_DIR)
print(f"\nrho cells: {len(rho_runs)}   gbm cells: {len(gbm_runs)}")

## 5. Analysis

### 5.1 Inventory and certificate audit

`holds` must be 1.00 everywhere. `tightness` = measured / bound is the price of
a worst-case argument; a very small value means the bound is loose, which is a
finding to report honestly rather than hide.

In [ ]:
import collections
runs = load_runs(OUT_DIR)
assert runs, f"no run records in {OUT_DIR}"
print(f"{len(runs)} run records\n")

by = collections.Counter((r["dataset"], r.get("teacher_kind", "mlp"), r["rho"])
                         for r in runs)
for (d, t, rho), n in sorted(by.items()):
    print(f"  {d:7s} teacher={t:4s} rho={rho:<5g} seeds={n}")

agg = collections.defaultdict(list)
for r in runs:
    c = r["students"]["gas_pce"].get("certificate")
    if c:
        agg[(r["dataset"], r["rho"])].append(c)

print(f"\n{'dataset':9s} {'rho':>5s} {'eta_mu':>8s} {'Delta':>8s} "
      f"{'measured':>9s} {'tight':>7s} {'holds':>6s}")

def _holdrate(cs):
    """Mean over cells where the bound was testable; '--' if none were."""
    t = [c["bound_holds"] for c in cs if c.get("bound_holds") is not None]
    return f"{np.mean(t):.2f}" if t else "--"

for (d, rho), cs in sorted(agg.items()):
    f = lambda k: np.nanmean([c[k] if np.isfinite(c[k]) else np.nan for c in cs])
    print(f"{d:9s} {rho:5g} {f('eta_mu'):8.3f} {f('bound_delta'):8.3f} "
          f"{f('measured_l1'):9.3f} {f('tightness'):7.3f} "
          f"{_holdrate(cs):>6s}")

# bound_holds is None where eta_mu >= 1 (Theorem 1 assumes eta < 1): such a
# cell is untested, not failed, and np.mean over a list holding None raises.
bad  = [r["cell"] for r in runs
        if r["students"]["gas_pce"]["certificate"].get("bound_holds") is False]
viol = [r["cell"] for r in runs
        if r["students"]["gas_pce"]["certificate"].get("precondition_violated")]
print(f"\ncertificate violations: {len(bad)}   "
      f"(eta_mu >= 1, not tested: {len(viol)})")
if bad:
    print("  ^ a BUG, not a result:", bad[:5])

### 5.2 The headline question (OP1): does label agreement predict explanation error?

If the two were tightly coupled, the prediction-level fidelity numbers reported
throughout the transparent-distillation literature would already control
explanation error, and the certificate would be redundant.

In [ ]:
from scipy.stats import spearmanr

agree, eta = [], []
for r in runs:
    for s in r["students"].values():
        fm = s.get("fidelity_mu")
        if fm and np.isfinite(fm["eta"]) and np.isfinite(fm["top1_agreement"]):
            agree.append(fm["top1_agreement"]); eta.append(fm["eta"])

print(f"n = {len(agree)} (dataset x student x seed)")
print(f"Spearman(top-1 agreement, eta_mu) = {spearmanr(agree, eta).statistic:.3f}")
print("\nStrongly negative -> agreement does track the certified residual.")
print("Weak -> the literature's fidelity numbers do not control explanation")
print("error, which is this paper's central empirical claim. Report either.")

### 5.3 Failure-mode diagnostics (§6 of the report)

**(ii) Off-manifold divergence** — the `eta_mu − eta_nu` gap at ρ = 0 is
Proposition 3 made visible. **(i) Truncation floor** — if `eta_mu` stops
improving as λ falls, the limit is truncation, not regularisation.
**(iv) Attribution splitting** — near-duplicate features make individual total
effects wobble across seeds while their sum stays stable.

In [ ]:
gap = collections.defaultdict(list)
for r in runs:
    g = r["students"]["gas_pce"]
    gap[(r["dataset"], r["rho"])].append(
        g["fidelity_mu"]["eta"] - g["fidelity_nu"]["eta"])
print("(ii) off-manifold gap")
print(f"  {'dataset':9s} {'rho':>5s} {'eta_mu - eta_nu':>17s}")
for k, v in sorted(gap.items()):
    print(f"  {k[0]:9s} {k[1]:5g} {np.nanmean(v):17.4f}")

print("\n(i) truncation floor along the lambda path")
for d in ("bank", "credit", "gas", "musk"):
    cur = [r["pce_frontier"] for r in runs if r["dataset"] == d
           and abs(r["rho"] - cfg.rho) < 1e-9 and r.get("pce_frontier")]
    if not cur:
        continue
    n = min(len(c) for c in cur)
    e = np.median([[p["eta_mu"] for p in c[:n]] for c in cur], axis=0)
    u = np.median([[p["active_components"] for p in c[:n]] for c in cur], axis=0)
    print(f"  {d:9s} eta_mu {e[0]:.3f} (|U|={u[0]:.0f}) -> {e[-1]:.3f} "
          f"(|U|={u[-1]:.0f})   last step {abs(e[-1]-e[-2]):.4f}")

print("\n(iv) seed stability of total effects")
for d in ("bank", "credit", "gas", "musk"):
    te = [np.array(r["students"]["gas_pce"]["total_effects"]) for r in runs
          if r["dataset"] == d and abs(r["rho"] - cfg.rho) < 1e-9
          and "total_effects" in r["students"]["gas_pce"]]
    if len(te) < 2:
        continue
    T = np.vstack(te)
    top5 = [np.sort(t)[::-1][:5].sum() for t in T]
    print(f"  {d:9s} seeds={T.shape[0]:2d}  per-feature sd="
          f"{T.std(axis=0, ddof=1).mean():.4f}  top-5 mass sd="
          f"{np.std(top5, ddof=1):.4f}")

## 6. Emit the LaTeX tables and figures

These are the files `paper/main.tex` `\input`s. Nothing in the report is typed
by hand.

In [ ]:
pathlib.Path("paper/results").mkdir(parents=True, exist_ok=True)
pathlib.Path("figures").mkdir(parents=True, exist_ok=True)

written = emit_all_tables(runs, "paper/results", rho=cfg.rho, teacher="mlp")
figs = emit_all_figures(runs, "figures", rho=cfg.rho, teacher="mlp")
for w in written:
    print("table :", w)
for f in figs:
    print("figure:", f)

print("\n--- tab_certificate.tex ---")
print(pathlib.Path("paper/results/tab_certificate.tex").read_text())

## 7. Package the artefacts

Unzip into `paper/` in the report directory and recompile.

In [ ]:
import zipfile
with zipfile.ZipFile("pact_artifacts.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in pathlib.Path("paper/results").glob("*.tex"):
        z.write(p, f"results/{p.name}")
    for p in pathlib.Path("figures").glob("*.pdf"):
        z.write(p, f"figures/{p.name}")
print("pact_artifacts.zip:", zipfile.ZipFile("pact_artifacts.zip").namelist())

if IN_COLAB:
    from google.colab import files
    files.download("pact_artifacts.zip")

---
### Before you submit

- [ ] `QUICK = False` and the full grid has run (10 seeds, 50 epochs, 4 datasets)
- [ ] `holds` is 1.00 for every dataset in §5.1
- [ ] no placeholder boxes left in the compiled PDF
- [ ] report ≤ 8 pages excluding references; Part A ≤ 3 pages
- [ ] the AI-use statement in `main.tex` matches what you actually did
- [ ] filename starts with `Surname_Name`